In [2]:
# Research review: Cross-asset variant: DF1 2007-2011, DF2 2012-2019, DF3 2020-2022, DF4 2023 onward. Preserved source claims are historical; clean-kernel reproduction is pending. Record target date and the availability of Close(t) used for ratio reconstruction.
"""
Cross-Asset Forecasting — TCN / Conv1D-LSTM / ConvLSTM-style, corrected.

Run this once per asset by changing ASSET below. Same script, same fixes,
for Gold, Crude Oil, Forex, and NIFTY 50 — that's the point of parameterizing
it instead of hand-copying per asset like the original notebook did.

Fixes applied vs. the original notebook (cells 20/21/22), all discussed and
agreed on earlier in this project:

  1. TCN: dilation schedule trimmed from [1,2,4,8] (~61-day receptive field)
     to [1,2,4] (~29 days) to match LOOKBACK=25, and GlobalAveragePooling1D
     (which averaged away recency) replaced with taking the last timestep.
  2. Conv1D-LSTM / ConvLSTM-style: added dilation to the Conv1D layers
     (receptive field ~5 days -> ~15 days with a 3-layer dilated stack),
     and BatchNormalization replaced with LayerNormalization (robust to
     shuffle=False + heavily overlapping sliding-window batches).
  3. Data loading is now self-contained per run: DATA_PATH is actually read
     with pd.read_csv() every time (previously set but unused dead code).
  4. Missing/unusable values in the raw source are removed via dropna after
     numeric coercion — confirmed as the intended, disclosed cleaning
     approach, not an artifact of how the data was queried. Gaps in the
     resulting date index (up to several weeks in Gold/Forex/Crude Oil) are
     expected and are handled by the history-aware sequence builder below,
     not treated as an error.
  5. Chronological train/val/test split happens BEFORE scaling, and the
     MinMaxScaler is fit on the training portion of each regime only.
  6. Regime boundaries are year-based (2007-2011 / 2012-2019 / 2020-2022 /
     2023-present), which works unmodified across all four assets despite
     their slightly different start dates (Gold/Crude/Forex ~Jan 2007,
     NIFTY Sept 2007) and their common end date (~Nov 2025).
"""

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, LSTM, Dense, Dropout, Add, Activation, LayerNormalization, Lambda
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# 1. CONFIG — pick the asset for this run
# ============================================================

ASSET_CONFIGS = {
    "Gold":      {"path": "gold_2007_to_present.csv",       "date_range": "2007-01-02 -> 2025-11-09"},
    "Crude Oil": {"path": "crude_oil_2007_to_present.csv",  "date_range": "2007-01-02 -> 2025-11-09"},
    "Forex":     {"path": "forex_usd_inr_2007_to_present.csv", "date_range": "2007-01-01 -> 2025-11-10"},
    "NIFTY 50":  {"path": "nifty50_2007_to_now.csv",        "date_range": "2007-09-17 -> 2025-11-25"},
}

ASSET = "Gold"   # <-- change this (and re-run) for each of the four assets
DATA_PATH = ASSET_CONFIGS[ASSET]["path"]

LOOKBACK = 25
FEATURES = ["Open", "High", "Low", "Close"]
EPOCHS = 100
BATCH_SIZE = 32
VAL_FRACTION = 0.15   # of each regime's pre-test data
TEST_FRACTION = 0.15  # of each regime

REGIMES = {
    "DF1": (2007, 2011),
    "DF2": (2012, 2019),
    "DF3": (2020, 2022),
    "DF4": (2023, None),   # None = open-ended, up to the max date actually in the file
}

# ============================================================
# 2. DATA LOADING — always reloaded fresh, never inherited from a stale df
# ============================================================

def load_asset_data(data_path):
    df = pd.read_csv(data_path)
    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
    df = df.dropna(subset=["Date"]).sort_values("Date").drop_duplicates(subset=["Date"], keep="first")

    for col in FEATURES:
        if col not in df.columns:
            raise ValueError(f"Expected column '{col}' not found. Columns present: {df.columns.tolist()}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    before = len(df)
    df = df.dropna(subset=FEATURES).reset_index(drop=True)   # remove rows with unusable/missing values
    removed = before - len(df)

    print(f"[{ASSET}] Loaded {data_path}: {len(df)} rows kept ({removed} rows removed for missing/invalid values)")
    print(f"[{ASSET}] Date range in file: {df['Date'].min().date()} -> {df['Date'].max().date()}")
    return df


def regime_slice(df, start_year, end_year):
    mask = df["Date"].dt.year >= start_year
    if end_year is not None:
        mask &= df["Date"].dt.year <= end_year
    return df[mask].reset_index(drop=True)


# ============================================================
# 3. TARGET CONSTRUCTION — next-observation High/Low as a ratio to Close
# ============================================================

def add_ratio_targets(regime_df):
    regime_df = regime_df.copy()
    regime_df["Target_High_Ratio"] = regime_df["High"].shift(-1) / regime_df["Close"]
    regime_df["Target_Low_Ratio"] = regime_df["Low"].shift(-1) / regime_df["Close"]
    return regime_df.dropna(subset=["Target_High_Ratio", "Target_Low_Ratio"]).reset_index(drop=True)


# ============================================================
# 4. CHRONOLOGICAL SPLIT + SCALING (fit on train only) + HISTORY-AWARE SEQUENCES
# ============================================================

def chronological_split(regime_df):
    n = len(regime_df)
    n_test = int(n * TEST_FRACTION)
    n_trainval = n - n_test
    n_val = int(n_trainval * VAL_FRACTION)
    n_train = n_trainval - n_val

    train_df = regime_df.iloc[:n_train].reset_index(drop=True)
    val_df = regime_df.iloc[n_train:n_train + n_val].reset_index(drop=True)
    test_df = regime_df.iloc[n_train + n_val:].reset_index(drop=True)
    return train_df, val_df, test_df


def create_sequences_with_history(history_X, history_close, split_X, split_y, split_close, lookback):
    """Builds sequences for a split using the tail of the PRECEDING split as
    lookback history, so the first `lookback` rows of val/test aren't wasted."""
    combined_X = np.concatenate([history_X[-lookback:], split_X], axis=0)
    X_seq, y_seq, close_seq = [], [], []
    for i in range(len(split_X)):
        X_seq.append(combined_X[i:i + lookback])
        y_seq.append(split_y[i])
        close_seq.append(split_close[i])
    return np.array(X_seq), np.array(y_seq), np.array(close_seq)


def prepare_regime(regime_df):
    train_df, val_df, test_df = chronological_split(regime_df)

    scaler = MinMaxScaler()
    scaler.fit(train_df[FEATURES])   # train-only fit — no leakage from val/test

    train_X = scaler.transform(train_df[FEATURES])
    val_X = scaler.transform(val_df[FEATURES])
    test_X = scaler.transform(test_df[FEATURES])

    train_y = train_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    val_y = val_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    test_y = test_df[["Target_High_Ratio", "Target_Low_Ratio"]].values

    # Plain (non-history) sequences for train:
    X_train, y_train = [], []
    for i in range(LOOKBACK, len(train_X)):
        X_train.append(train_X[i - LOOKBACK:i])
        y_train.append(train_y[i])
    X_train, y_train = np.array(X_train), np.array(y_train)
    close_train = train_df["Close"].values[LOOKBACK:]

    # History-aware sequences for val/test:
    X_val, y_val, close_val = create_sequences_with_history(
        train_X, train_df["Close"].values, val_X, val_y, val_df["Close"].values, LOOKBACK
    )
    X_test, y_test, close_test = create_sequences_with_history(
        np.concatenate([train_X, val_X], axis=0), np.concatenate([train_df["Close"].values, val_df["Close"].values]),
        test_X, test_y, test_df["Close"].values, LOOKBACK
    )

    return {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val, "close_val": close_val,
        "X_test": X_test, "y_test": y_test, "close_test": close_test,
        "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    }


# ============================================================
# 5. CORRECTED MODEL ARCHITECTURES
# ============================================================

TCN_FILTERS = 64
CONV_FILTERS = 64
KERNEL_SIZE = 3
TCN_DILATIONS = [1, 2, 4]      # trimmed from [1,2,4,8]; receptive field ~29 days, matches LOOKBACK=25
DROPOUT = 0.20
DENSE_UNITS = 16
LEARNING_RATE = 0.001


def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    residual = x
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding="causal",
               dilation_rate=dilation_rate, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding="causal",
               dilation_rate=dilation_rate, activation=None)(x)
    x = Dropout(dropout_rate)(x)
    if residual.shape[-1] != filters:
        residual = Conv1D(filters=filters, kernel_size=1, padding="same")(residual)
    x = Add()([x, residual])
    return Activation("relu")(x)


def build_tcn_model(input_shape):
    inputs = Input(shape=input_shape)
    x = inputs
    for dilation in TCN_DILATIONS:
        x = tcn_residual_block(x, TCN_FILTERS, KERNEL_SIZE, dilation, DROPOUT)
    x = Lambda(lambda t: t[:, -1, :])(x)          # last timestep, not GlobalAveragePooling1D
    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="TCN")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse", metrics=["mae"])
    return model


def build_conv1d_lstm_model(input_shape):
    inputs = Input(shape=input_shape)
    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=1)(inputs)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=2)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=4)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(DROPOUT)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="Conv1D-LSTM")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    return model


def build_convlstm_style_model(input_shape):
    # Same fixes as Conv1D-LSTM; kept as a separate configuration (ordering/
    # depth style) so the two remain a genuine ablation rather than a
    # duplicate, as agreed on earlier.
    inputs = Input(shape=input_shape)
    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=1, activation=None)(inputs)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=2, activation=None)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=4, activation=None)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(DROPOUT)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="ConvLSTM-style")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    return model


MODEL_BUILDERS = {
    "TCN": build_tcn_model,
    "Conv1D-LSTM": build_conv1d_lstm_model,
    "ConvLSTM-style": build_convlstm_style_model,
}

# ============================================================
# 6. METRICS — computed on reconstructed absolute price, not the ratio scale
# ============================================================

def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}


def train_and_evaluate(model_name, regime_name, data):
    builder = MODEL_BUILDERS[model_name]
    model = builder(input_shape=(LOOKBACK, len(FEATURES)))

    early_stop = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
    model.fit(
        data["X_train"], data["y_train"],
        validation_data=(data["X_val"], data["y_val"]),
        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=False,
        callbacks=[early_stop], verbose=0,
    )

    pred_ratio = model.predict(data["X_test"], verbose=0)
    pred_high = pred_ratio[:, 0] * data["close_test"]
    pred_low = pred_ratio[:, 1] * data["close_test"]

    true_high = data["y_test"][:, 0] * data["close_test"]
    true_low = data["y_test"][:, 1] * data["close_test"]

    metrics_high = compute_metrics(true_high, pred_high)
    metrics_low = compute_metrics(true_low, pred_low)

    return {
        "Asset": ASSET, "Model": model_name, "Regime": regime_name,
        "R2_High": metrics_high["R2"], "R2_Low": metrics_low["R2"],
        "MAE_High": metrics_high["MAE"], "MAE_Low": metrics_low["MAE"],
        "RMSE_High": metrics_high["RMSE"], "RMSE_Low": metrics_low["RMSE"],
        "MAPE_High": metrics_high["MAPE"], "MAPE_Low": metrics_low["MAPE"],
        "n_train": data["n_train"], "n_val": data["n_val"], "n_test": data["n_test"],
    }


# ============================================================
# 7. MAIN LOOP — all regimes x all three corrected architectures
# ============================================================

if __name__ == "__main__":
    df = load_asset_data(DATA_PATH)
    results = []

    for regime_name, (start_year, end_year) in REGIMES.items():
        regime_df = regime_slice(df, start_year, end_year)
        regime_df = add_ratio_targets(regime_df)

        min_required = LOOKBACK + 20
        if len(regime_df) <= min_required:
            print(f"[{ASSET}] Skipping {regime_name}: only {len(regime_df)} usable rows (< {min_required}).")
            continue

        data = prepare_regime(regime_df)
        print(f"\n[{ASSET}] {regime_name}: train={data['n_train']}, val={data['n_val']}, test={data['n_test']}")

        for model_name in MODEL_BUILDERS:
            print(f"  Training {model_name}...")
            result = train_and_evaluate(model_name, regime_name, data)
            results.append(result)
            print(f"    R2 High/Low: {result['R2_High']:.3f} / {result['R2_Low']:.3f}")

    results_df = pd.DataFrame(results)
    print(f"\n=== {ASSET} — Full Results ===")
    print(results_df.to_string(index=False))
    results_df.to_csv(f"{ASSET.replace(' ', '_').lower()}_tcn_convlstm_results.csv", index=False)

[Gold] Loaded gold_2007_to_present.csv: 4745 rows kept (0 rows removed for missing/invalid values)
[Gold] Date range in file: 2007-01-02 -> 2025-11-09

[Gold] DF1: train=911, val=160, test=189
  Training TCN...
    R2 High/Low: -2.034 / -1.715
  Training Conv1D-LSTM...
    R2 High/Low: 0.904 / 0.861
  Training ConvLSTM-style...
    R2 High/Low: 0.970 / 0.962

[Gold] DF2: train=1451, val=256, test=301
  Training TCN...
    R2 High/Low: 0.974 / 0.987
  Training Conv1D-LSTM...
    R2 High/Low: 0.993 / 0.810
  Training ConvLSTM-style...
    R2 High/Low: 0.921 / 0.961

[Gold] DF3: train=546, val=96, test=113
  Training TCN...
    R2 High/Low: 0.915 / 0.931
  Training Conv1D-LSTM...
    R2 High/Low: -0.176 / 0.397
  Training ConvLSTM-style...
    R2 High/Low: 0.754 / 0.804

[Gold] DF4: train=520, val=91, test=107
  Training TCN...
    R2 High/Low: -7.022 / -8.200
  Training Conv1D-LSTM...
    R2 High/Low: 0.986 / 0.946
  Training ConvLSTM-style...
    R2 High/Low: 0.974 / 0.868

=== Gold — F

In [3]:
"""
Cross-Asset Forecasting — TCN / Conv1D-LSTM / ConvLSTM-style, corrected.

Run this once per asset by changing ASSET below. Same script, same fixes,
for Gold, Crude Oil, Forex, and NIFTY 50 — that's the point of parameterizing
it instead of hand-copying per asset like the original notebook did.

Fixes applied vs. the original notebook (cells 20/21/22), all discussed and
agreed on earlier in this project:

  1. TCN: dilation schedule trimmed from [1,2,4,8] (~61-day receptive field)
     to [1,2,4] (~29 days) to match LOOKBACK=25, and GlobalAveragePooling1D
     (which averaged away recency) replaced with taking the last timestep.
  2. Conv1D-LSTM / ConvLSTM-style: added dilation to the Conv1D layers
     (receptive field ~5 days -> ~15 days with a 3-layer dilated stack),
     and BatchNormalization replaced with LayerNormalization (robust to
     shuffle=False + heavily overlapping sliding-window batches).
  3. Data loading is now self-contained per run: DATA_PATH is actually read
     with pd.read_csv() every time (previously set but unused dead code).
  4. Missing/unusable values in the raw source are removed via dropna after
     numeric coercion — confirmed as the intended, disclosed cleaning
     approach, not an artifact of how the data was queried. Gaps in the
     resulting date index (up to several weeks in Gold/Forex/Crude Oil) are
     expected and are handled by the history-aware sequence builder below,
     not treated as an error.
  5. Chronological train/val/test split happens BEFORE scaling, and the
     MinMaxScaler is fit on the training portion of each regime only.
  6. Regime boundaries are year-based (2007-2011 / 2012-2019 / 2020-2022 /
     2023-present), which works unmodified across all four assets despite
     their slightly different start dates (Gold/Crude/Forex ~Jan 2007,
     NIFTY Sept 2007) and their common end date (~Nov 2025).
"""

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, LSTM, Dense, Dropout, Add, Activation, LayerNormalization, Lambda
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# 1. CONFIG — pick the asset for this run
# ============================================================

ASSET_CONFIGS = {
    "Gold":      {"path": "gold_2007_to_present.csv",       "date_range": "2007-01-02 -> 2025-11-09"},
    "Crude Oil": {"path": "crude_oil_2007_to_present.csv",  "date_range": "2007-01-02 -> 2025-11-09"},
    "Forex":     {"path": "forex_usd_inr_2007_to_present.csv", "date_range": "2007-01-01 -> 2025-11-10"},
    "NIFTY 50":  {"path": "nifty50_2007_to_now.csv",        "date_range": "2007-09-17 -> 2025-11-25"},
}

ASSET = "Crude Oil"   # <-- change this (and re-run) for each of the four assets
DATA_PATH = ASSET_CONFIGS[ASSET]["path"]

LOOKBACK = 25
FEATURES = ["Open", "High", "Low", "Close"]
EPOCHS = 100
BATCH_SIZE = 32
VAL_FRACTION = 0.15   # of each regime's pre-test data
TEST_FRACTION = 0.15  # of each regime

REGIMES = {
    "DF1": (2007, 2011),
    "DF2": (2012, 2019),
    "DF3": (2020, 2022),
    "DF4": (2023, None),   # None = open-ended, up to the max date actually in the file
}

# ============================================================
# 2. DATA LOADING — always reloaded fresh, never inherited from a stale df
# ============================================================

def load_asset_data(data_path):
    df = pd.read_csv(data_path)
    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
    df = df.dropna(subset=["Date"]).sort_values("Date").drop_duplicates(subset=["Date"], keep="first")

    for col in FEATURES:
        if col not in df.columns:
            raise ValueError(f"Expected column '{col}' not found. Columns present: {df.columns.tolist()}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    before = len(df)
    df = df.dropna(subset=FEATURES).reset_index(drop=True)   # remove rows with unusable/missing values
    removed = before - len(df)

    print(f"[{ASSET}] Loaded {data_path}: {len(df)} rows kept ({removed} rows removed for missing/invalid values)")
    print(f"[{ASSET}] Date range in file: {df['Date'].min().date()} -> {df['Date'].max().date()}")
    return df


def regime_slice(df, start_year, end_year):
    mask = df["Date"].dt.year >= start_year
    if end_year is not None:
        mask &= df["Date"].dt.year <= end_year
    return df[mask].reset_index(drop=True)


# ============================================================
# 3. TARGET CONSTRUCTION — next-observation High/Low as a ratio to Close
# ============================================================

def add_ratio_targets(regime_df):
    regime_df = regime_df.copy()
    regime_df["Target_High_Ratio"] = regime_df["High"].shift(-1) / regime_df["Close"]
    regime_df["Target_Low_Ratio"] = regime_df["Low"].shift(-1) / regime_df["Close"]
    return regime_df.dropna(subset=["Target_High_Ratio", "Target_Low_Ratio"]).reset_index(drop=True)


# ============================================================
# 4. CHRONOLOGICAL SPLIT + SCALING (fit on train only) + HISTORY-AWARE SEQUENCES
# ============================================================

def chronological_split(regime_df):
    n = len(regime_df)
    n_test = int(n * TEST_FRACTION)
    n_trainval = n - n_test
    n_val = int(n_trainval * VAL_FRACTION)
    n_train = n_trainval - n_val

    train_df = regime_df.iloc[:n_train].reset_index(drop=True)
    val_df = regime_df.iloc[n_train:n_train + n_val].reset_index(drop=True)
    test_df = regime_df.iloc[n_train + n_val:].reset_index(drop=True)
    return train_df, val_df, test_df


def create_sequences_with_history(history_X, history_close, split_X, split_y, split_close, lookback):
    """Builds sequences for a split using the tail of the PRECEDING split as
    lookback history, so the first `lookback` rows of val/test aren't wasted."""
    combined_X = np.concatenate([history_X[-lookback:], split_X], axis=0)
    X_seq, y_seq, close_seq = [], [], []
    for i in range(len(split_X)):
        X_seq.append(combined_X[i:i + lookback])
        y_seq.append(split_y[i])
        close_seq.append(split_close[i])
    return np.array(X_seq), np.array(y_seq), np.array(close_seq)


def prepare_regime(regime_df):
    train_df, val_df, test_df = chronological_split(regime_df)

    scaler = MinMaxScaler()
    scaler.fit(train_df[FEATURES])   # train-only fit — no leakage from val/test

    train_X = scaler.transform(train_df[FEATURES])
    val_X = scaler.transform(val_df[FEATURES])
    test_X = scaler.transform(test_df[FEATURES])

    train_y = train_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    val_y = val_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    test_y = test_df[["Target_High_Ratio", "Target_Low_Ratio"]].values

    # Plain (non-history) sequences for train:
    X_train, y_train = [], []
    for i in range(LOOKBACK, len(train_X)):
        X_train.append(train_X[i - LOOKBACK:i])
        y_train.append(train_y[i])
    X_train, y_train = np.array(X_train), np.array(y_train)
    close_train = train_df["Close"].values[LOOKBACK:]

    # History-aware sequences for val/test:
    X_val, y_val, close_val = create_sequences_with_history(
        train_X, train_df["Close"].values, val_X, val_y, val_df["Close"].values, LOOKBACK
    )
    X_test, y_test, close_test = create_sequences_with_history(
        np.concatenate([train_X, val_X], axis=0), np.concatenate([train_df["Close"].values, val_df["Close"].values]),
        test_X, test_y, test_df["Close"].values, LOOKBACK
    )

    return {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val, "close_val": close_val,
        "X_test": X_test, "y_test": y_test, "close_test": close_test,
        "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    }


# ============================================================
# 5. CORRECTED MODEL ARCHITECTURES
# ============================================================

TCN_FILTERS = 64
CONV_FILTERS = 64
KERNEL_SIZE = 3
TCN_DILATIONS = [1, 2, 4]      # trimmed from [1,2,4,8]; receptive field ~29 days, matches LOOKBACK=25
DROPOUT = 0.20
DENSE_UNITS = 16
LEARNING_RATE = 0.001


def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    residual = x
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding="causal",
               dilation_rate=dilation_rate, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding="causal",
               dilation_rate=dilation_rate, activation=None)(x)
    x = Dropout(dropout_rate)(x)
    if residual.shape[-1] != filters:
        residual = Conv1D(filters=filters, kernel_size=1, padding="same")(residual)
    x = Add()([x, residual])
    return Activation("relu")(x)


def build_tcn_model(input_shape):
    inputs = Input(shape=input_shape)
    x = inputs
    for dilation in TCN_DILATIONS:
        x = tcn_residual_block(x, TCN_FILTERS, KERNEL_SIZE, dilation, DROPOUT)
    x = Lambda(lambda t: t[:, -1, :])(x)          # last timestep, not GlobalAveragePooling1D
    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="TCN")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse", metrics=["mae"])
    return model


def build_conv1d_lstm_model(input_shape):
    inputs = Input(shape=input_shape)
    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=1)(inputs)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=2)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=4)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(DROPOUT)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="Conv1D-LSTM")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    return model


def build_convlstm_style_model(input_shape):
    # Same fixes as Conv1D-LSTM; kept as a separate configuration (ordering/
    # depth style) so the two remain a genuine ablation rather than a
    # duplicate, as agreed on earlier.
    inputs = Input(shape=input_shape)
    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=1, activation=None)(inputs)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=2, activation=None)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=4, activation=None)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(DROPOUT)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="ConvLSTM-style")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    return model


MODEL_BUILDERS = {
    "TCN": build_tcn_model,
    "Conv1D-LSTM": build_conv1d_lstm_model,
    "ConvLSTM-style": build_convlstm_style_model,
}

# ============================================================
# 6. METRICS — computed on reconstructed absolute price, not the ratio scale
# ============================================================

def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}


def train_and_evaluate(model_name, regime_name, data):
    builder = MODEL_BUILDERS[model_name]
    model = builder(input_shape=(LOOKBACK, len(FEATURES)))

    early_stop = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
    model.fit(
        data["X_train"], data["y_train"],
        validation_data=(data["X_val"], data["y_val"]),
        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=False,
        callbacks=[early_stop], verbose=0,
    )

    pred_ratio = model.predict(data["X_test"], verbose=0)
    pred_high = pred_ratio[:, 0] * data["close_test"]
    pred_low = pred_ratio[:, 1] * data["close_test"]

    true_high = data["y_test"][:, 0] * data["close_test"]
    true_low = data["y_test"][:, 1] * data["close_test"]

    metrics_high = compute_metrics(true_high, pred_high)
    metrics_low = compute_metrics(true_low, pred_low)

    return {
        "Asset": ASSET, "Model": model_name, "Regime": regime_name,
        "R2_High": metrics_high["R2"], "R2_Low": metrics_low["R2"],
        "MAE_High": metrics_high["MAE"], "MAE_Low": metrics_low["MAE"],
        "RMSE_High": metrics_high["RMSE"], "RMSE_Low": metrics_low["RMSE"],
        "MAPE_High": metrics_high["MAPE"], "MAPE_Low": metrics_low["MAPE"],
        "n_train": data["n_train"], "n_val": data["n_val"], "n_test": data["n_test"],
    }


# ============================================================
# 7. MAIN LOOP — all regimes x all three corrected architectures
# ============================================================

if __name__ == "__main__":
    df = load_asset_data(DATA_PATH)
    results = []

    for regime_name, (start_year, end_year) in REGIMES.items():
        regime_df = regime_slice(df, start_year, end_year)
        regime_df = add_ratio_targets(regime_df)

        min_required = LOOKBACK + 20
        if len(regime_df) <= min_required:
            print(f"[{ASSET}] Skipping {regime_name}: only {len(regime_df)} usable rows (< {min_required}).")
            continue

        data = prepare_regime(regime_df)
        print(f"\n[{ASSET}] {regime_name}: train={data['n_train']}, val={data['n_val']}, test={data['n_test']}")

        for model_name in MODEL_BUILDERS:
            print(f"  Training {model_name}...")
            result = train_and_evaluate(model_name, regime_name, data)
            results.append(result)
            print(f"    R2 High/Low: {result['R2_High']:.3f} / {result['R2_Low']:.3f}")

    results_df = pd.DataFrame(results)
    print(f"\n=== {ASSET} — Full Results ===")
    print(results_df.to_string(index=False))
    results_df.to_csv(f"{ASSET.replace(' ', '_').lower()}_tcn_convlstm_results.csv", index=False)

[Crude Oil] Loaded crude_oil_2007_to_present.csv: 4746 rows kept (0 rows removed for missing/invalid values)
[Crude Oil] Date range in file: 2007-01-02 -> 2025-11-09

[Crude Oil] DF1: train=911, val=160, test=189
  Training TCN...
    R2 High/Low: 0.960 / 0.942
  Training Conv1D-LSTM...
    R2 High/Low: 0.899 / 0.838
  Training ConvLSTM-style...
    R2 High/Low: 0.956 / 0.909

[Crude Oil] DF2: train=1452, val=256, test=301
  Training TCN...
    R2 High/Low: 0.961 / 0.949
  Training Conv1D-LSTM...
    R2 High/Low: 0.868 / 0.934
  Training ConvLSTM-style...
    R2 High/Low: 0.927 / 0.951

[Crude Oil] DF3: train=546, val=96, test=113
  Training TCN...
    R2 High/Low: 0.115 / 0.550
  Training Conv1D-LSTM...
    R2 High/Low: 0.933 / 0.942
  Training ConvLSTM-style...
    R2 High/Low: 0.952 / 0.910

[Crude Oil] DF4: train=520, val=91, test=107
  Training TCN...
    R2 High/Low: -0.962 / 0.351
  Training Conv1D-LSTM...
    R2 High/Low: 0.754 / 0.827
  Training ConvLSTM-style...
    R2 High/L

In [6]:
"""
Cross-Asset Forecasting — TCN / Conv1D-LSTM / ConvLSTM-style, corrected.

Run this once per asset by changing ASSET below. Same script, same fixes,
for Gold, Crude Oil, Forex, and NIFTY 50 — that's the point of parameterizing
it instead of hand-copying per asset like the original notebook did.

Fixes applied vs. the original notebook (cells 20/21/22), all discussed and
agreed on earlier in this project:

  1. TCN: dilation schedule trimmed from [1,2,4,8] (~61-day receptive field)
     to [1,2,4] (~29 days) to match LOOKBACK=25, and GlobalAveragePooling1D
     (which averaged away recency) replaced with taking the last timestep.
  2. Conv1D-LSTM / ConvLSTM-style: added dilation to the Conv1D layers
     (receptive field ~5 days -> ~15 days with a 3-layer dilated stack),
     and BatchNormalization replaced with LayerNormalization (robust to
     shuffle=False + heavily overlapping sliding-window batches).
  3. Data loading is now self-contained per run: DATA_PATH is actually read
     with pd.read_csv() every time (previously set but unused dead code).
  4. Missing/unusable values in the raw source are removed via dropna after
     numeric coercion — confirmed as the intended, disclosed cleaning
     approach, not an artifact of how the data was queried. Gaps in the
     resulting date index (up to several weeks in Gold/Forex/Crude Oil) are
     expected and are handled by the history-aware sequence builder below,
     not treated as an error.
  5. Chronological train/val/test split happens BEFORE scaling, and the
     MinMaxScaler is fit on the training portion of each regime only.
  6. Regime boundaries are year-based (2007-2011 / 2012-2019 / 2020-2022 /
     2023-present), which works unmodified across all four assets despite
     their slightly different start dates (Gold/Crude/Forex ~Jan 2007,
     NIFTY Sept 2007) and their common end date (~Nov 2025).
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, LSTM, Dense, Dropout, Add, Activation, LayerNormalization, Lambda
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# 1. CONFIG — pick the asset for this run
# ============================================================

ASSET_CONFIGS = {
    "Gold":      {"path": "gold_2007_to_present.csv",       "date_range": "2007-01-02 -> 2025-11-09"},
    "Crude Oil": {"path": "crude_oil_2007_to_present.csv",  "date_range": "2007-01-02 -> 2025-11-09"},
    "Forex":     {"path": "forex_usd_inr_2007_to_present.csv", "date_range": "2007-01-01 -> 2025-11-10"},
    "NIFTY 50":  {"path": "nifty50_2007_to_now.csv",        "date_range": "2007-09-17 -> 2025-11-25"},
}

ASSET = "Gold"   # <-- change this (and re-run) for each of the four assets
DATA_PATH = ASSET_CONFIGS[ASSET]["path"]

LOOKBACK = 25
FEATURES = ["Open", "High", "Low", "Close"]
EPOCHS = 100
BATCH_SIZE = 32
VAL_FRACTION = 0.15   # of each regime's pre-test data
TEST_FRACTION = 0.15  # of each regime

REGIMES = {
    "DF1": (2007, 2011),
    "DF2": (2012, 2019),
    "DF3": (2020, 2022),
    "DF4": (2023, None),   # None = open-ended, up to the max date actually in the file
}

# ============================================================
# 2. DATA LOADING — always reloaded fresh, never inherited from a stale df
# ============================================================

def load_asset_data(data_path):
    df = pd.read_csv(data_path)
    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
    df = df.dropna(subset=["Date"]).sort_values("Date").drop_duplicates(subset=["Date"], keep="first")

    for col in FEATURES:
        if col not in df.columns:
            raise ValueError(f"Expected column '{col}' not found. Columns present: {df.columns.tolist()}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    before = len(df)
    df = df.dropna(subset=FEATURES).reset_index(drop=True)   # remove rows with unusable/missing values
    removed = before - len(df)

    print(f"[{ASSET}] Loaded {data_path}: {len(df)} rows kept ({removed} rows removed for missing/invalid values)")
    print(f"[{ASSET}] Date range in file: {df['Date'].min().date()} -> {df['Date'].max().date()}")
    return df


def regime_slice(df, start_year, end_year):
    mask = df["Date"].dt.year >= start_year
    if end_year is not None:
        mask &= df["Date"].dt.year <= end_year
    return df[mask].reset_index(drop=True)


# ============================================================
# 3. TARGET CONSTRUCTION — next-observation High/Low as a ratio to Close
# ============================================================

def add_ratio_targets(regime_df):
    regime_df = regime_df.copy()
    regime_df["Target_High_Ratio"] = regime_df["High"].shift(-1) / regime_df["Close"]
    regime_df["Target_Low_Ratio"] = regime_df["Low"].shift(-1) / regime_df["Close"]
    return regime_df.dropna(subset=["Target_High_Ratio", "Target_Low_Ratio"]).reset_index(drop=True)


# ============================================================
# 4. CHRONOLOGICAL SPLIT + SCALING (fit on train only) + HISTORY-AWARE SEQUENCES
# ============================================================

def chronological_split(regime_df):
    n = len(regime_df)
    n_test = int(n * TEST_FRACTION)
    n_trainval = n - n_test
    n_val = int(n_trainval * VAL_FRACTION)
    n_train = n_trainval - n_val

    train_df = regime_df.iloc[:n_train].reset_index(drop=True)
    val_df = regime_df.iloc[n_train:n_train + n_val].reset_index(drop=True)
    test_df = regime_df.iloc[n_train + n_val:].reset_index(drop=True)
    return train_df, val_df, test_df


def create_sequences_with_history(history_X, history_close, split_X, split_y, split_close, lookback):
    """Builds sequences for a split using the tail of the PRECEDING split as
    lookback history, so the first `lookback` rows of val/test aren't wasted."""
    combined_X = np.concatenate([history_X[-lookback:], split_X], axis=0)
    X_seq, y_seq, close_seq = [], [], []
    for i in range(len(split_X)):
        X_seq.append(combined_X[i:i + lookback])
        y_seq.append(split_y[i])
        close_seq.append(split_close[i])
    return np.array(X_seq), np.array(y_seq), np.array(close_seq)


def prepare_regime(regime_df):
    train_df, val_df, test_df = chronological_split(regime_df)

    scaler = MinMaxScaler()
    scaler.fit(train_df[FEATURES])   # train-only fit — no leakage from val/test

    train_X = scaler.transform(train_df[FEATURES])
    val_X = scaler.transform(val_df[FEATURES])
    test_X = scaler.transform(test_df[FEATURES])

    train_y = train_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    val_y = val_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    test_y = test_df[["Target_High_Ratio", "Target_Low_Ratio"]].values

    # Plain (non-history) sequences for train:
    X_train, y_train = [], []
    for i in range(LOOKBACK, len(train_X)):
        X_train.append(train_X[i - LOOKBACK:i])
        y_train.append(train_y[i])
    X_train, y_train = np.array(X_train), np.array(y_train)
    close_train = train_df["Close"].values[LOOKBACK:]

    # History-aware sequences for val/test:
    X_val, y_val, close_val = create_sequences_with_history(
        train_X, train_df["Close"].values, val_X, val_y, val_df["Close"].values, LOOKBACK
    )
    X_test, y_test, close_test = create_sequences_with_history(
        np.concatenate([train_X, val_X], axis=0), np.concatenate([train_df["Close"].values, val_df["Close"].values]),
        test_X, test_y, test_df["Close"].values, LOOKBACK
    )

    return {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val, "close_val": close_val,
        "X_test": X_test, "y_test": y_test, "close_test": close_test,
        "dates_test": test_df["Date"].values,
        "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    }


# ============================================================
# 5. CORRECTED MODEL ARCHITECTURES
# ============================================================

TCN_FILTERS = 64
CONV_FILTERS = 64
KERNEL_SIZE = 3
TCN_DILATIONS = [1, 2, 4]      # trimmed from [1,2,4,8]; receptive field ~29 days, matches LOOKBACK=25
DROPOUT = 0.20
DENSE_UNITS = 16
LEARNING_RATE = 0.001


def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    residual = x
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding="causal",
               dilation_rate=dilation_rate, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding="causal",
               dilation_rate=dilation_rate, activation=None)(x)
    x = Dropout(dropout_rate)(x)
    if residual.shape[-1] != filters:
        residual = Conv1D(filters=filters, kernel_size=1, padding="same")(residual)
    x = Add()([x, residual])
    return Activation("relu")(x)


def build_tcn_model(input_shape):
    inputs = Input(shape=input_shape)
    x = inputs
    for dilation in TCN_DILATIONS:
        x = tcn_residual_block(x, TCN_FILTERS, KERNEL_SIZE, dilation, DROPOUT)
    x = Lambda(lambda t: t[:, -1, :])(x)          # last timestep, not GlobalAveragePooling1D
    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="TCN")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse", metrics=["mae"])
    return model


def build_conv1d_lstm_model(input_shape):
    inputs = Input(shape=input_shape)
    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=1)(inputs)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=2)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=4)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(DROPOUT)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="Conv1D-LSTM")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    return model


def build_convlstm_style_model(input_shape):
    # Same fixes as Conv1D-LSTM; kept as a separate configuration (ordering/
    # depth style) so the two remain a genuine ablation rather than a
    # duplicate, as agreed on earlier.
    inputs = Input(shape=input_shape)
    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=1, activation=None)(inputs)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=2, activation=None)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=4, activation=None)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(DROPOUT)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="ConvLSTM-style")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    return model


MODEL_BUILDERS = {
    "TCN": build_tcn_model,
    "Conv1D-LSTM": build_conv1d_lstm_model,
    "ConvLSTM-style": build_convlstm_style_model,
}

# ============================================================
# 6. METRICS — computed on reconstructed absolute price, not the ratio scale
# ============================================================

def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}


def train_and_evaluate(model_name, regime_name, data):
    builder = MODEL_BUILDERS[model_name]
    model = builder(input_shape=(LOOKBACK, len(FEATURES)))

    early_stop = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
    model.fit(
        data["X_train"], data["y_train"],
        validation_data=(data["X_val"], data["y_val"]),
        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=False,
        callbacks=[early_stop], verbose=0,
    )

    pred_ratio = model.predict(data["X_test"], verbose=0)
    pred_high = pred_ratio[:, 0] * data["close_test"]
    pred_low = pred_ratio[:, 1] * data["close_test"]

    true_high = data["y_test"][:, 0] * data["close_test"]
    true_low = data["y_test"][:, 1] * data["close_test"]

    metrics_high = compute_metrics(true_high, pred_high)
    metrics_low = compute_metrics(true_low, pred_low)

    result = {
        "Asset": ASSET, "Model": model_name, "Regime": regime_name,
        "R2_High": metrics_high["R2"], "R2_Low": metrics_low["R2"],
        "MAE_High": metrics_high["MAE"], "MAE_Low": metrics_low["MAE"],
        "RMSE_High": metrics_high["RMSE"], "RMSE_Low": metrics_low["RMSE"],
        "MAPE_High": metrics_high["MAPE"], "MAPE_Low": metrics_low["MAPE"],
        "n_train": data["n_train"], "n_val": data["n_val"], "n_test": data["n_test"],
    }

    predictions = {
        "dates": data["dates_test"],
        "true_high": true_high, "pred_high": pred_high,
        "true_low": true_low, "pred_low": pred_low,
        "r2_high": metrics_high["R2"], "r2_low": metrics_low["R2"],
    }

    return result, predictions


REGIME_ORDER = ["DF1", "DF2", "DF3", "DF4"]


def plot_predicted_vs_actual_by_model(asset_name, all_predictions):
    """
    One figure PER MODEL (TCN, Conv1D-LSTM, ConvLSTM-style), with all
    available regimes (DF1..DF4) as subplots inside it — clubbed by model,
    not one figure per regime with all models crammed inside it.

    Each subplot shows actual vs. predicted High and Low over that regime's
    test period, with R² in the subplot title. Saves one PNG per model.
    """
    for model_name, regime_preds in all_predictions.items():
        regimes_present = [r for r in REGIME_ORDER if r in regime_preds]
        if not regimes_present:
            continue

        n = len(regimes_present)
        n_cols = 2 if n > 1 else 1
        n_rows = int(np.ceil(n / n_cols))

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 4.5 * n_rows), squeeze=False)
        fig.suptitle(f"{asset_name} — {model_name}: Predicted vs. Actual (all regimes)", fontsize=14, y=1.02)

        for idx, regime_name in enumerate(regimes_present):
            ax = axes[idx // n_cols][idx % n_cols]
            p = regime_preds[regime_name]

            ax.plot(p["dates"], p["true_high"], label="Actual High", color="tab:blue", linewidth=1.2)
            ax.plot(p["dates"], p["pred_high"], label="Predicted High", color="tab:blue", linewidth=1.0, linestyle="--")
            ax.plot(p["dates"], p["true_low"], label="Actual Low", color="tab:orange", linewidth=1.2)
            ax.plot(p["dates"], p["pred_low"], label="Predicted Low", color="tab:orange", linewidth=1.0, linestyle="--")

            ax.set_title(f"{regime_name}  (R\u00b2 High={p['r2_high']:.3f}, Low={p['r2_low']:.3f})", fontsize=11)
            ax.set_xlabel("Date")
            ax.set_ylabel("Price")
            ax.tick_params(axis="x", rotation=30)
            if idx == 0:
                ax.legend(fontsize=8, loc="best")

        # hide any unused subplot slots
        for idx in range(n, n_rows * n_cols):
            axes[idx // n_cols][idx % n_cols].axis("off")

        plt.tight_layout()
        filename = f"{asset_name.replace(' ', '_').lower()}_{model_name.replace(' ', '_').lower()}_predicted_vs_actual.png"
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {filename}")


def plot_r2_comparison_by_model(asset_name, results_df):
    """
    Other relevant plot: a single bar chart per model showing R² (High and
    Low) across DF1..DF4 side by side — a compact companion to the line
    plots above for quickly comparing regime difficulty within one model.
    """
    for model_name in results_df["Model"].unique():
        sub = results_df[results_df["Model"] == model_name].set_index("Regime").reindex(REGIME_ORDER).dropna(how="all")
        if sub.empty:
            continue

        fig, ax = plt.subplots(figsize=(7, 4))
        x = np.arange(len(sub))
        width = 0.35
        ax.bar(x - width / 2, sub["R2_High"], width, label="R² High", color="tab:blue")
        ax.bar(x + width / 2, sub["R2_Low"], width, label="R² Low", color="tab:orange")
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(sub.index)
        ax.set_ylabel("R²")
        ax.set_title(f"{asset_name} — {model_name}: R² by Regime")
        ax.legend()
        plt.tight_layout()
        filename = f"{asset_name.replace(' ', '_').lower()}_{model_name.replace(' ', '_').lower()}_r2_by_regime.png"
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {filename}")


# ============================================================
# 8. MAIN LOOP — all regimes x all three corrected architectures
# ============================================================

if __name__ == "__main__":
    df = load_asset_data(DATA_PATH)
    results = []
    # all_predictions[model_name][regime_name] = predictions dict — grouped by
    # MODEL first, so plotting can club all four regimes under one model,
    # not one figure per regime with models inside it.
    all_predictions = {model_name: {} for model_name in MODEL_BUILDERS}

    for regime_name, (start_year, end_year) in REGIMES.items():
        regime_df = regime_slice(df, start_year, end_year)
        regime_df = add_ratio_targets(regime_df)

        min_required = LOOKBACK + 20
        if len(regime_df) <= min_required:
            print(f"[{ASSET}] Skipping {regime_name}: only {len(regime_df)} usable rows (< {min_required}).")
            continue

        data = prepare_regime(regime_df)
        print(f"\n[{ASSET}] {regime_name}: train={data['n_train']}, val={data['n_val']}, test={data['n_test']}")

        for model_name in MODEL_BUILDERS:
            print(f"  Training {model_name}...")
            result, predictions = train_and_evaluate(model_name, regime_name, data)
            results.append(result)
            all_predictions[model_name][regime_name] = predictions
            print(f"    R2 High/Low: {result['R2_High']:.3f} / {result['R2_Low']:.3f}")

    results_df = pd.DataFrame(results)
    print(f"\n=== {ASSET} — Full Results ===")
    print(results_df.to_string(index=False))
    results_df.to_csv(f"{ASSET.replace(' ', '_').lower()}_tcn_convlstm_results.csv", index=False)

    plot_predicted_vs_actual_by_model(ASSET, all_predictions)
    plot_r2_comparison_by_model(ASSET, results_df)

[Gold] Loaded gold_2007_to_present.csv: 4745 rows kept (0 rows removed for missing/invalid values)
[Gold] Date range in file: 2007-01-02 -> 2025-11-09

[Gold] DF1: train=911, val=160, test=189
  Training TCN...
    R2 High/Low: 0.492 / -0.052
  Training Conv1D-LSTM...
    R2 High/Low: 0.899 / 0.890
  Training ConvLSTM-style...
    R2 High/Low: 0.889 / 0.959

[Gold] DF2: train=1451, val=256, test=301
  Training TCN...
    R2 High/Low: 0.992 / 0.978
  Training Conv1D-LSTM...
    R2 High/Low: 0.993 / 0.985
  Training ConvLSTM-style...
    R2 High/Low: 0.993 / 0.988

[Gold] DF3: train=546, val=96, test=113
  Training TCN...
    R2 High/Low: 0.792 / 0.693
  Training Conv1D-LSTM...
    R2 High/Low: 0.911 / 0.893
  Training ConvLSTM-style...
    R2 High/Low: 0.837 / 0.945

[Gold] DF4: train=520, val=91, test=107
  Training TCN...
    R2 High/Low: -26.160 / -20.978
  Training Conv1D-LSTM...
    R2 High/Low: 0.975 / 0.977
  Training ConvLSTM-style...
    R2 High/Low: 0.954 / 0.972

=== Gold — F

In [7]:
"""
Cross-Asset Forecasting — TCN / Conv1D-LSTM / ConvLSTM-style, corrected.

Run this once per asset by changing ASSET below. Same script, same fixes,
for Gold, Crude Oil, Forex, and NIFTY 50 — that's the point of parameterizing
it instead of hand-copying per asset like the original notebook did.

Fixes applied vs. the original notebook (cells 20/21/22), all discussed and
agreed on earlier in this project:

  1. TCN: dilation schedule trimmed from [1,2,4,8] (~61-day receptive field)
     to [1,2,4] (~29 days) to match LOOKBACK=25, and GlobalAveragePooling1D
     (which averaged away recency) replaced with taking the last timestep.
  2. Conv1D-LSTM / ConvLSTM-style: added dilation to the Conv1D layers
     (receptive field ~5 days -> ~15 days with a 3-layer dilated stack),
     and BatchNormalization replaced with LayerNormalization (robust to
     shuffle=False + heavily overlapping sliding-window batches).
  3. Data loading is now self-contained per run: DATA_PATH is actually read
     with pd.read_csv() every time (previously set but unused dead code).
  4. Missing/unusable values in the raw source are removed via dropna after
     numeric coercion — confirmed as the intended, disclosed cleaning
     approach, not an artifact of how the data was queried. Gaps in the
     resulting date index (up to several weeks in Gold/Forex/Crude Oil) are
     expected and are handled by the history-aware sequence builder below,
     not treated as an error.
  5. Chronological train/val/test split happens BEFORE scaling, and the
     MinMaxScaler is fit on the training portion of each regime only.
  6. Regime boundaries are year-based (2007-2011 / 2012-2019 / 2020-2022 /
     2023-present), which works unmodified across all four assets despite
     their slightly different start dates (Gold/Crude/Forex ~Jan 2007,
     NIFTY Sept 2007) and their common end date (~Nov 2025).
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, LSTM, Dense, Dropout, Add, Activation, LayerNormalization, Lambda
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# 1. CONFIG — pick the asset for this run
# ============================================================

ASSET_CONFIGS = {
    "Gold":      {"path": "gold_2007_to_present.csv",       "date_range": "2007-01-02 -> 2025-11-09"},
    "Crude Oil": {"path": "crude_oil_2007_to_present.csv",  "date_range": "2007-01-02 -> 2025-11-09"},
    "Forex":     {"path": "forex_usd_inr_2007_to_present.csv", "date_range": "2007-01-01 -> 2025-11-10"},
    "NIFTY 50":  {"path": "nifty50_2007_to_now.csv",        "date_range": "2007-09-17 -> 2025-11-25"},
}

ASSET = "Crude Oil"   # <-- change this (and re-run) for each of the four assets
DATA_PATH = ASSET_CONFIGS[ASSET]["path"]

LOOKBACK = 25
FEATURES = ["Open", "High", "Low", "Close"]
EPOCHS = 100
BATCH_SIZE = 32
VAL_FRACTION = 0.15   # of each regime's pre-test data
TEST_FRACTION = 0.15  # of each regime

REGIMES = {
    "DF1": (2007, 2011),
    "DF2": (2012, 2019),
    "DF3": (2020, 2022),
    "DF4": (2023, None),   # None = open-ended, up to the max date actually in the file
}

# ============================================================
# 2. DATA LOADING — always reloaded fresh, never inherited from a stale df
# ============================================================

def load_asset_data(data_path):
    df = pd.read_csv(data_path)
    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
    df = df.dropna(subset=["Date"]).sort_values("Date").drop_duplicates(subset=["Date"], keep="first")

    for col in FEATURES:
        if col not in df.columns:
            raise ValueError(f"Expected column '{col}' not found. Columns present: {df.columns.tolist()}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    before = len(df)
    df = df.dropna(subset=FEATURES).reset_index(drop=True)   # remove rows with unusable/missing values
    removed = before - len(df)

    print(f"[{ASSET}] Loaded {data_path}: {len(df)} rows kept ({removed} rows removed for missing/invalid values)")
    print(f"[{ASSET}] Date range in file: {df['Date'].min().date()} -> {df['Date'].max().date()}")
    return df


def regime_slice(df, start_year, end_year):
    mask = df["Date"].dt.year >= start_year
    if end_year is not None:
        mask &= df["Date"].dt.year <= end_year
    return df[mask].reset_index(drop=True)


# ============================================================
# 3. TARGET CONSTRUCTION — next-observation High/Low as a ratio to Close
# ============================================================

def add_ratio_targets(regime_df):
    regime_df = regime_df.copy()
    regime_df["Target_High_Ratio"] = regime_df["High"].shift(-1) / regime_df["Close"]
    regime_df["Target_Low_Ratio"] = regime_df["Low"].shift(-1) / regime_df["Close"]
    return regime_df.dropna(subset=["Target_High_Ratio", "Target_Low_Ratio"]).reset_index(drop=True)


# ============================================================
# 4. CHRONOLOGICAL SPLIT + SCALING (fit on train only) + HISTORY-AWARE SEQUENCES
# ============================================================

def chronological_split(regime_df):
    n = len(regime_df)
    n_test = int(n * TEST_FRACTION)
    n_trainval = n - n_test
    n_val = int(n_trainval * VAL_FRACTION)
    n_train = n_trainval - n_val

    train_df = regime_df.iloc[:n_train].reset_index(drop=True)
    val_df = regime_df.iloc[n_train:n_train + n_val].reset_index(drop=True)
    test_df = regime_df.iloc[n_train + n_val:].reset_index(drop=True)
    return train_df, val_df, test_df


def create_sequences_with_history(history_X, history_close, split_X, split_y, split_close, lookback):
    """Builds sequences for a split using the tail of the PRECEDING split as
    lookback history, so the first `lookback` rows of val/test aren't wasted."""
    combined_X = np.concatenate([history_X[-lookback:], split_X], axis=0)
    X_seq, y_seq, close_seq = [], [], []
    for i in range(len(split_X)):
        X_seq.append(combined_X[i:i + lookback])
        y_seq.append(split_y[i])
        close_seq.append(split_close[i])
    return np.array(X_seq), np.array(y_seq), np.array(close_seq)


def prepare_regime(regime_df):
    train_df, val_df, test_df = chronological_split(regime_df)

    scaler = MinMaxScaler()
    scaler.fit(train_df[FEATURES])   # train-only fit — no leakage from val/test

    train_X = scaler.transform(train_df[FEATURES])
    val_X = scaler.transform(val_df[FEATURES])
    test_X = scaler.transform(test_df[FEATURES])

    train_y = train_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    val_y = val_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    test_y = test_df[["Target_High_Ratio", "Target_Low_Ratio"]].values

    # Plain (non-history) sequences for train:
    X_train, y_train = [], []
    for i in range(LOOKBACK, len(train_X)):
        X_train.append(train_X[i - LOOKBACK:i])
        y_train.append(train_y[i])
    X_train, y_train = np.array(X_train), np.array(y_train)
    close_train = train_df["Close"].values[LOOKBACK:]

    # History-aware sequences for val/test:
    X_val, y_val, close_val = create_sequences_with_history(
        train_X, train_df["Close"].values, val_X, val_y, val_df["Close"].values, LOOKBACK
    )
    X_test, y_test, close_test = create_sequences_with_history(
        np.concatenate([train_X, val_X], axis=0), np.concatenate([train_df["Close"].values, val_df["Close"].values]),
        test_X, test_y, test_df["Close"].values, LOOKBACK
    )

    return {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val, "close_val": close_val,
        "X_test": X_test, "y_test": y_test, "close_test": close_test,
        "dates_test": test_df["Date"].values,
        "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    }


# ============================================================
# 5. CORRECTED MODEL ARCHITECTURES
# ============================================================

TCN_FILTERS = 64
CONV_FILTERS = 64
KERNEL_SIZE = 3
TCN_DILATIONS = [1, 2, 4]      # trimmed from [1,2,4,8]; receptive field ~29 days, matches LOOKBACK=25
DROPOUT = 0.20
DENSE_UNITS = 16
LEARNING_RATE = 0.001


def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    residual = x
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding="causal",
               dilation_rate=dilation_rate, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding="causal",
               dilation_rate=dilation_rate, activation=None)(x)
    x = Dropout(dropout_rate)(x)
    if residual.shape[-1] != filters:
        residual = Conv1D(filters=filters, kernel_size=1, padding="same")(residual)
    x = Add()([x, residual])
    return Activation("relu")(x)


def build_tcn_model(input_shape):
    inputs = Input(shape=input_shape)
    x = inputs
    for dilation in TCN_DILATIONS:
        x = tcn_residual_block(x, TCN_FILTERS, KERNEL_SIZE, dilation, DROPOUT)
    x = Lambda(lambda t: t[:, -1, :])(x)          # last timestep, not GlobalAveragePooling1D
    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="TCN")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse", metrics=["mae"])
    return model


def build_conv1d_lstm_model(input_shape):
    inputs = Input(shape=input_shape)
    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=1)(inputs)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=2)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=4)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(DROPOUT)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="Conv1D-LSTM")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    return model


def build_convlstm_style_model(input_shape):
    # Same fixes as Conv1D-LSTM; kept as a separate configuration (ordering/
    # depth style) so the two remain a genuine ablation rather than a
    # duplicate, as agreed on earlier.
    inputs = Input(shape=input_shape)
    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=1, activation=None)(inputs)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=2, activation=None)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=4, activation=None)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(DROPOUT)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="ConvLSTM-style")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    return model


MODEL_BUILDERS = {
    "TCN": build_tcn_model,
    "Conv1D-LSTM": build_conv1d_lstm_model,
    "ConvLSTM-style": build_convlstm_style_model,
}

# ============================================================
# 6. METRICS — computed on reconstructed absolute price, not the ratio scale
# ============================================================

def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}


def train_and_evaluate(model_name, regime_name, data):
    builder = MODEL_BUILDERS[model_name]
    model = builder(input_shape=(LOOKBACK, len(FEATURES)))

    early_stop = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
    model.fit(
        data["X_train"], data["y_train"],
        validation_data=(data["X_val"], data["y_val"]),
        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=False,
        callbacks=[early_stop], verbose=0,
    )

    pred_ratio = model.predict(data["X_test"], verbose=0)
    pred_high = pred_ratio[:, 0] * data["close_test"]
    pred_low = pred_ratio[:, 1] * data["close_test"]

    true_high = data["y_test"][:, 0] * data["close_test"]
    true_low = data["y_test"][:, 1] * data["close_test"]

    metrics_high = compute_metrics(true_high, pred_high)
    metrics_low = compute_metrics(true_low, pred_low)

    result = {
        "Asset": ASSET, "Model": model_name, "Regime": regime_name,
        "R2_High": metrics_high["R2"], "R2_Low": metrics_low["R2"],
        "MAE_High": metrics_high["MAE"], "MAE_Low": metrics_low["MAE"],
        "RMSE_High": metrics_high["RMSE"], "RMSE_Low": metrics_low["RMSE"],
        "MAPE_High": metrics_high["MAPE"], "MAPE_Low": metrics_low["MAPE"],
        "n_train": data["n_train"], "n_val": data["n_val"], "n_test": data["n_test"],
    }

    predictions = {
        "dates": data["dates_test"],
        "true_high": true_high, "pred_high": pred_high,
        "true_low": true_low, "pred_low": pred_low,
        "r2_high": metrics_high["R2"], "r2_low": metrics_low["R2"],
    }

    return result, predictions


REGIME_ORDER = ["DF1", "DF2", "DF3", "DF4"]


def plot_predicted_vs_actual_by_model(asset_name, all_predictions):
    """
    One figure PER MODEL (TCN, Conv1D-LSTM, ConvLSTM-style), with all
    available regimes (DF1..DF4) as subplots inside it — clubbed by model,
    not one figure per regime with all models crammed inside it.

    Each subplot shows actual vs. predicted High and Low over that regime's
    test period, with R² in the subplot title. Saves one PNG per model.
    """
    for model_name, regime_preds in all_predictions.items():
        regimes_present = [r for r in REGIME_ORDER if r in regime_preds]
        if not regimes_present:
            continue

        n = len(regimes_present)
        n_cols = 2 if n > 1 else 1
        n_rows = int(np.ceil(n / n_cols))

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 4.5 * n_rows), squeeze=False)
        fig.suptitle(f"{asset_name} — {model_name}: Predicted vs. Actual (all regimes)", fontsize=14, y=1.02)

        for idx, regime_name in enumerate(regimes_present):
            ax = axes[idx // n_cols][idx % n_cols]
            p = regime_preds[regime_name]

            ax.plot(p["dates"], p["true_high"], label="Actual High", color="tab:blue", linewidth=1.2)
            ax.plot(p["dates"], p["pred_high"], label="Predicted High", color="tab:blue", linewidth=1.0, linestyle="--")
            ax.plot(p["dates"], p["true_low"], label="Actual Low", color="tab:orange", linewidth=1.2)
            ax.plot(p["dates"], p["pred_low"], label="Predicted Low", color="tab:orange", linewidth=1.0, linestyle="--")

            ax.set_title(f"{regime_name}  (R\u00b2 High={p['r2_high']:.3f}, Low={p['r2_low']:.3f})", fontsize=11)
            ax.set_xlabel("Date")
            ax.set_ylabel("Price")
            ax.tick_params(axis="x", rotation=30)
            if idx == 0:
                ax.legend(fontsize=8, loc="best")

        # hide any unused subplot slots
        for idx in range(n, n_rows * n_cols):
            axes[idx // n_cols][idx % n_cols].axis("off")

        plt.tight_layout()
        filename = f"{asset_name.replace(' ', '_').lower()}_{model_name.replace(' ', '_').lower()}_predicted_vs_actual.png"
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {filename}")


def plot_r2_comparison_by_model(asset_name, results_df):
    """
    Other relevant plot: a single bar chart per model showing R² (High and
    Low) across DF1..DF4 side by side — a compact companion to the line
    plots above for quickly comparing regime difficulty within one model.
    """
    for model_name in results_df["Model"].unique():
        sub = results_df[results_df["Model"] == model_name].set_index("Regime").reindex(REGIME_ORDER).dropna(how="all")
        if sub.empty:
            continue

        fig, ax = plt.subplots(figsize=(7, 4))
        x = np.arange(len(sub))
        width = 0.35
        ax.bar(x - width / 2, sub["R2_High"], width, label="R² High", color="tab:blue")
        ax.bar(x + width / 2, sub["R2_Low"], width, label="R² Low", color="tab:orange")
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(sub.index)
        ax.set_ylabel("R²")
        ax.set_title(f"{asset_name} — {model_name}: R² by Regime")
        ax.legend()
        plt.tight_layout()
        filename = f"{asset_name.replace(' ', '_').lower()}_{model_name.replace(' ', '_').lower()}_r2_by_regime.png"
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {filename}")


# ============================================================
# 8. MAIN LOOP — all regimes x all three corrected architectures
# ============================================================

if __name__ == "__main__":
    df = load_asset_data(DATA_PATH)
    results = []
    # all_predictions[model_name][regime_name] = predictions dict — grouped by
    # MODEL first, so plotting can club all four regimes under one model,
    # not one figure per regime with models inside it.
    all_predictions = {model_name: {} for model_name in MODEL_BUILDERS}

    for regime_name, (start_year, end_year) in REGIMES.items():
        regime_df = regime_slice(df, start_year, end_year)
        regime_df = add_ratio_targets(regime_df)

        min_required = LOOKBACK + 20
        if len(regime_df) <= min_required:
            print(f"[{ASSET}] Skipping {regime_name}: only {len(regime_df)} usable rows (< {min_required}).")
            continue

        data = prepare_regime(regime_df)
        print(f"\n[{ASSET}] {regime_name}: train={data['n_train']}, val={data['n_val']}, test={data['n_test']}")

        for model_name in MODEL_BUILDERS:
            print(f"  Training {model_name}...")
            result, predictions = train_and_evaluate(model_name, regime_name, data)
            results.append(result)
            all_predictions[model_name][regime_name] = predictions
            print(f"    R2 High/Low: {result['R2_High']:.3f} / {result['R2_Low']:.3f}")

    results_df = pd.DataFrame(results)
    print(f"\n=== {ASSET} — Full Results ===")
    print(results_df.to_string(index=False))
    results_df.to_csv(f"{ASSET.replace(' ', '_').lower()}_tcn_convlstm_results.csv", index=False)

    plot_predicted_vs_actual_by_model(ASSET, all_predictions)
    plot_r2_comparison_by_model(ASSET, results_df)

[Crude Oil] Loaded crude_oil_2007_to_present.csv: 4746 rows kept (0 rows removed for missing/invalid values)
[Crude Oil] Date range in file: 2007-01-02 -> 2025-11-09

[Crude Oil] DF1: train=911, val=160, test=189
  Training TCN...
    R2 High/Low: 0.968 / 0.957
  Training Conv1D-LSTM...
    R2 High/Low: 0.908 / 0.957
  Training ConvLSTM-style...
    R2 High/Low: 0.537 / -0.269

[Crude Oil] DF2: train=1452, val=256, test=301
  Training TCN...
    R2 High/Low: 0.954 / 0.950
  Training Conv1D-LSTM...
    R2 High/Low: 0.958 / 0.953
  Training ConvLSTM-style...
    R2 High/Low: 0.888 / 0.952

[Crude Oil] DF3: train=546, val=96, test=113
  Training TCN...
    R2 High/Low: -1.792 / -1.284
  Training Conv1D-LSTM...
    R2 High/Low: 0.900 / 0.880
  Training ConvLSTM-style...
    R2 High/Low: 0.955 / 0.931

[Crude Oil] DF4: train=520, val=91, test=107
  Training TCN...
    R2 High/Low: 0.619 / 0.656
  Training Conv1D-LSTM...
    R2 High/Low: 0.805 / 0.689
  Training ConvLSTM-style...
    R2 High

In [8]:
"""
Cross-Asset Forecasting — TCN / Conv1D-LSTM / ConvLSTM-style, corrected.

Run this once per asset by changing ASSET below. Same script, same fixes,
for Gold, Crude Oil, Forex, and NIFTY 50 — that's the point of parameterizing
it instead of hand-copying per asset like the original notebook did.

Fixes applied vs. the original notebook (cells 20/21/22), all discussed and
agreed on earlier in this project:

  1. TCN: dilation schedule trimmed from [1,2,4,8] (~61-day receptive field)
     to [1,2,4] (~29 days) to match LOOKBACK=25, and GlobalAveragePooling1D
     (which averaged away recency) replaced with taking the last timestep.
  2. Conv1D-LSTM / ConvLSTM-style: added dilation to the Conv1D layers
     (receptive field ~5 days -> ~15 days with a 3-layer dilated stack),
     and BatchNormalization replaced with LayerNormalization (robust to
     shuffle=False + heavily overlapping sliding-window batches).
  3. Data loading is now self-contained per run: DATA_PATH is actually read
     with pd.read_csv() every time (previously set but unused dead code).
  4. Missing/unusable values in the raw source are removed via dropna after
     numeric coercion — confirmed as the intended, disclosed cleaning
     approach, not an artifact of how the data was queried. Gaps in the
     resulting date index (up to several weeks in Gold/Forex/Crude Oil) are
     expected and are handled by the history-aware sequence builder below,
     not treated as an error.
  5. Chronological train/val/test split happens BEFORE scaling, and the
     MinMaxScaler is fit on the training portion of each regime only.
  6. Regime boundaries are year-based (2007-2011 / 2012-2019 / 2020-2022 /
     2023-present), which works unmodified across all four assets despite
     their slightly different start dates (Gold/Crude/Forex ~Jan 2007,
     NIFTY Sept 2007) and their common end date (~Nov 2025).
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, LSTM, Dense, Dropout, Add, Activation, LayerNormalization, Lambda
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# 1. CONFIG — pick the asset for this run
# ============================================================

ASSET_CONFIGS = {
    "Gold":      {"path": "gold_2007_to_present.csv",       "date_range": "2007-01-02 -> 2025-11-09"},
    "Crude Oil": {"path": "crude_oil_2007_to_present.csv",  "date_range": "2007-01-02 -> 2025-11-09"},
    "Forex":     {"path": "forex_usd_inr_2007_to_present.csv", "date_range": "2007-01-01 -> 2025-11-10"},
    "NIFTY 50":  {"path": "nifty50_2007_to_now.csv",        "date_range": "2007-09-17 -> 2025-11-25"},
}

ASSET = "Forex"   # <-- change this (and re-run) for each of the four assets
DATA_PATH = ASSET_CONFIGS[ASSET]["path"]

LOOKBACK = 25
FEATURES = ["Open", "High", "Low", "Close"]
EPOCHS = 100
BATCH_SIZE = 32
VAL_FRACTION = 0.15   # of each regime's pre-test data
TEST_FRACTION = 0.15  # of each regime

REGIMES = {
    "DF1": (2007, 2011),
    "DF2": (2012, 2019),
    "DF3": (2020, 2022),
    "DF4": (2023, None),   # None = open-ended, up to the max date actually in the file
}

# ============================================================
# 2. DATA LOADING — always reloaded fresh, never inherited from a stale df
# ============================================================

def load_asset_data(data_path):
    df = pd.read_csv(data_path)
    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
    df = df.dropna(subset=["Date"]).sort_values("Date").drop_duplicates(subset=["Date"], keep="first")

    for col in FEATURES:
        if col not in df.columns:
            raise ValueError(f"Expected column '{col}' not found. Columns present: {df.columns.tolist()}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    before = len(df)
    df = df.dropna(subset=FEATURES).reset_index(drop=True)   # remove rows with unusable/missing values
    removed = before - len(df)

    print(f"[{ASSET}] Loaded {data_path}: {len(df)} rows kept ({removed} rows removed for missing/invalid values)")
    print(f"[{ASSET}] Date range in file: {df['Date'].min().date()} -> {df['Date'].max().date()}")
    return df


def regime_slice(df, start_year, end_year):
    mask = df["Date"].dt.year >= start_year
    if end_year is not None:
        mask &= df["Date"].dt.year <= end_year
    return df[mask].reset_index(drop=True)


# ============================================================
# 3. TARGET CONSTRUCTION — next-observation High/Low as a ratio to Close
# ============================================================

def add_ratio_targets(regime_df):
    regime_df = regime_df.copy()
    regime_df["Target_High_Ratio"] = regime_df["High"].shift(-1) / regime_df["Close"]
    regime_df["Target_Low_Ratio"] = regime_df["Low"].shift(-1) / regime_df["Close"]
    return regime_df.dropna(subset=["Target_High_Ratio", "Target_Low_Ratio"]).reset_index(drop=True)


# ============================================================
# 4. CHRONOLOGICAL SPLIT + SCALING (fit on train only) + HISTORY-AWARE SEQUENCES
# ============================================================

def chronological_split(regime_df):
    n = len(regime_df)
    n_test = int(n * TEST_FRACTION)
    n_trainval = n - n_test
    n_val = int(n_trainval * VAL_FRACTION)
    n_train = n_trainval - n_val

    train_df = regime_df.iloc[:n_train].reset_index(drop=True)
    val_df = regime_df.iloc[n_train:n_train + n_val].reset_index(drop=True)
    test_df = regime_df.iloc[n_train + n_val:].reset_index(drop=True)
    return train_df, val_df, test_df


def create_sequences_with_history(history_X, history_close, split_X, split_y, split_close, lookback):
    """Builds sequences for a split using the tail of the PRECEDING split as
    lookback history, so the first `lookback` rows of val/test aren't wasted."""
    combined_X = np.concatenate([history_X[-lookback:], split_X], axis=0)
    X_seq, y_seq, close_seq = [], [], []
    for i in range(len(split_X)):
        X_seq.append(combined_X[i:i + lookback])
        y_seq.append(split_y[i])
        close_seq.append(split_close[i])
    return np.array(X_seq), np.array(y_seq), np.array(close_seq)


def prepare_regime(regime_df):
    train_df, val_df, test_df = chronological_split(regime_df)

    scaler = MinMaxScaler()
    scaler.fit(train_df[FEATURES])   # train-only fit — no leakage from val/test

    train_X = scaler.transform(train_df[FEATURES])
    val_X = scaler.transform(val_df[FEATURES])
    test_X = scaler.transform(test_df[FEATURES])

    train_y = train_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    val_y = val_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    test_y = test_df[["Target_High_Ratio", "Target_Low_Ratio"]].values

    # Plain (non-history) sequences for train:
    X_train, y_train = [], []
    for i in range(LOOKBACK, len(train_X)):
        X_train.append(train_X[i - LOOKBACK:i])
        y_train.append(train_y[i])
    X_train, y_train = np.array(X_train), np.array(y_train)
    close_train = train_df["Close"].values[LOOKBACK:]

    # History-aware sequences for val/test:
    X_val, y_val, close_val = create_sequences_with_history(
        train_X, train_df["Close"].values, val_X, val_y, val_df["Close"].values, LOOKBACK
    )
    X_test, y_test, close_test = create_sequences_with_history(
        np.concatenate([train_X, val_X], axis=0), np.concatenate([train_df["Close"].values, val_df["Close"].values]),
        test_X, test_y, test_df["Close"].values, LOOKBACK
    )

    return {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val, "close_val": close_val,
        "X_test": X_test, "y_test": y_test, "close_test": close_test,
        "dates_test": test_df["Date"].values,
        "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    }


# ============================================================
# 5. CORRECTED MODEL ARCHITECTURES
# ============================================================

TCN_FILTERS = 64
CONV_FILTERS = 64
KERNEL_SIZE = 3
TCN_DILATIONS = [1, 2, 4]      # trimmed from [1,2,4,8]; receptive field ~29 days, matches LOOKBACK=25
DROPOUT = 0.20
DENSE_UNITS = 16
LEARNING_RATE = 0.001


def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    residual = x
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding="causal",
               dilation_rate=dilation_rate, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding="causal",
               dilation_rate=dilation_rate, activation=None)(x)
    x = Dropout(dropout_rate)(x)
    if residual.shape[-1] != filters:
        residual = Conv1D(filters=filters, kernel_size=1, padding="same")(residual)
    x = Add()([x, residual])
    return Activation("relu")(x)


def build_tcn_model(input_shape):
    inputs = Input(shape=input_shape)
    x = inputs
    for dilation in TCN_DILATIONS:
        x = tcn_residual_block(x, TCN_FILTERS, KERNEL_SIZE, dilation, DROPOUT)
    x = Lambda(lambda t: t[:, -1, :])(x)          # last timestep, not GlobalAveragePooling1D
    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="TCN")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse", metrics=["mae"])
    return model


def build_conv1d_lstm_model(input_shape):
    inputs = Input(shape=input_shape)
    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=1)(inputs)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=2)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=4)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(DROPOUT)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="Conv1D-LSTM")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    return model


def build_convlstm_style_model(input_shape):
    # Same fixes as Conv1D-LSTM; kept as a separate configuration (ordering/
    # depth style) so the two remain a genuine ablation rather than a
    # duplicate, as agreed on earlier.
    inputs = Input(shape=input_shape)
    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=1, activation=None)(inputs)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=2, activation=None)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=4, activation=None)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(DROPOUT)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="ConvLSTM-style")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    return model


MODEL_BUILDERS = {
    "TCN": build_tcn_model,
    "Conv1D-LSTM": build_conv1d_lstm_model,
    "ConvLSTM-style": build_convlstm_style_model,
}

# ============================================================
# 6. METRICS — computed on reconstructed absolute price, not the ratio scale
# ============================================================

def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}


def train_and_evaluate(model_name, regime_name, data):
    builder = MODEL_BUILDERS[model_name]
    model = builder(input_shape=(LOOKBACK, len(FEATURES)))

    early_stop = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
    model.fit(
        data["X_train"], data["y_train"],
        validation_data=(data["X_val"], data["y_val"]),
        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=False,
        callbacks=[early_stop], verbose=0,
    )

    pred_ratio = model.predict(data["X_test"], verbose=0)
    pred_high = pred_ratio[:, 0] * data["close_test"]
    pred_low = pred_ratio[:, 1] * data["close_test"]

    true_high = data["y_test"][:, 0] * data["close_test"]
    true_low = data["y_test"][:, 1] * data["close_test"]

    metrics_high = compute_metrics(true_high, pred_high)
    metrics_low = compute_metrics(true_low, pred_low)

    result = {
        "Asset": ASSET, "Model": model_name, "Regime": regime_name,
        "R2_High": metrics_high["R2"], "R2_Low": metrics_low["R2"],
        "MAE_High": metrics_high["MAE"], "MAE_Low": metrics_low["MAE"],
        "RMSE_High": metrics_high["RMSE"], "RMSE_Low": metrics_low["RMSE"],
        "MAPE_High": metrics_high["MAPE"], "MAPE_Low": metrics_low["MAPE"],
        "n_train": data["n_train"], "n_val": data["n_val"], "n_test": data["n_test"],
    }

    predictions = {
        "dates": data["dates_test"],
        "true_high": true_high, "pred_high": pred_high,
        "true_low": true_low, "pred_low": pred_low,
        "r2_high": metrics_high["R2"], "r2_low": metrics_low["R2"],
    }

    return result, predictions


REGIME_ORDER = ["DF1", "DF2", "DF3", "DF4"]


def plot_predicted_vs_actual_by_model(asset_name, all_predictions):
    """
    One figure PER MODEL (TCN, Conv1D-LSTM, ConvLSTM-style), with all
    available regimes (DF1..DF4) as subplots inside it — clubbed by model,
    not one figure per regime with all models crammed inside it.

    Each subplot shows actual vs. predicted High and Low over that regime's
    test period, with R² in the subplot title. Saves one PNG per model.
    """
    for model_name, regime_preds in all_predictions.items():
        regimes_present = [r for r in REGIME_ORDER if r in regime_preds]
        if not regimes_present:
            continue

        n = len(regimes_present)
        n_cols = 2 if n > 1 else 1
        n_rows = int(np.ceil(n / n_cols))

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 4.5 * n_rows), squeeze=False)
        fig.suptitle(f"{asset_name} — {model_name}: Predicted vs. Actual (all regimes)", fontsize=14, y=1.02)

        for idx, regime_name in enumerate(regimes_present):
            ax = axes[idx // n_cols][idx % n_cols]
            p = regime_preds[regime_name]

            ax.plot(p["dates"], p["true_high"], label="Actual High", color="tab:blue", linewidth=1.2)
            ax.plot(p["dates"], p["pred_high"], label="Predicted High", color="tab:blue", linewidth=1.0, linestyle="--")
            ax.plot(p["dates"], p["true_low"], label="Actual Low", color="tab:orange", linewidth=1.2)
            ax.plot(p["dates"], p["pred_low"], label="Predicted Low", color="tab:orange", linewidth=1.0, linestyle="--")

            ax.set_title(f"{regime_name}  (R\u00b2 High={p['r2_high']:.3f}, Low={p['r2_low']:.3f})", fontsize=11)
            ax.set_xlabel("Date")
            ax.set_ylabel("Price")
            ax.tick_params(axis="x", rotation=30)
            if idx == 0:
                ax.legend(fontsize=8, loc="best")

        # hide any unused subplot slots
        for idx in range(n, n_rows * n_cols):
            axes[idx // n_cols][idx % n_cols].axis("off")

        plt.tight_layout()
        filename = f"{asset_name.replace(' ', '_').lower()}_{model_name.replace(' ', '_').lower()}_predicted_vs_actual.png"
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {filename}")


def plot_r2_comparison_by_model(asset_name, results_df):
    """
    Other relevant plot: a single bar chart per model showing R² (High and
    Low) across DF1..DF4 side by side — a compact companion to the line
    plots above for quickly comparing regime difficulty within one model.
    """
    for model_name in results_df["Model"].unique():
        sub = results_df[results_df["Model"] == model_name].set_index("Regime").reindex(REGIME_ORDER).dropna(how="all")
        if sub.empty:
            continue

        fig, ax = plt.subplots(figsize=(7, 4))
        x = np.arange(len(sub))
        width = 0.35
        ax.bar(x - width / 2, sub["R2_High"], width, label="R² High", color="tab:blue")
        ax.bar(x + width / 2, sub["R2_Low"], width, label="R² Low", color="tab:orange")
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(sub.index)
        ax.set_ylabel("R²")
        ax.set_title(f"{asset_name} — {model_name}: R² by Regime")
        ax.legend()
        plt.tight_layout()
        filename = f"{asset_name.replace(' ', '_').lower()}_{model_name.replace(' ', '_').lower()}_r2_by_regime.png"
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {filename}")


# ============================================================
# 8. MAIN LOOP — all regimes x all three corrected architectures
# ============================================================

if __name__ == "__main__":
    df = load_asset_data(DATA_PATH)
    results = []
    # all_predictions[model_name][regime_name] = predictions dict — grouped by
    # MODEL first, so plotting can club all four regimes under one model,
    # not one figure per regime with models inside it.
    all_predictions = {model_name: {} for model_name in MODEL_BUILDERS}

    for regime_name, (start_year, end_year) in REGIMES.items():
        regime_df = regime_slice(df, start_year, end_year)
        regime_df = add_ratio_targets(regime_df)

        min_required = LOOKBACK + 20
        if len(regime_df) <= min_required:
            print(f"[{ASSET}] Skipping {regime_name}: only {len(regime_df)} usable rows (< {min_required}).")
            continue

        data = prepare_regime(regime_df)
        print(f"\n[{ASSET}] {regime_name}: train={data['n_train']}, val={data['n_val']}, test={data['n_test']}")

        for model_name in MODEL_BUILDERS:
            print(f"  Training {model_name}...")
            result, predictions = train_and_evaluate(model_name, regime_name, data)
            results.append(result)
            all_predictions[model_name][regime_name] = predictions
            print(f"    R2 High/Low: {result['R2_High']:.3f} / {result['R2_Low']:.3f}")

    results_df = pd.DataFrame(results)
    print(f"\n=== {ASSET} — Full Results ===")
    print(results_df.to_string(index=False))
    results_df.to_csv(f"{ASSET.replace(' ', '_').lower()}_tcn_convlstm_results.csv", index=False)

    plot_predicted_vs_actual_by_model(ASSET, all_predictions)
    plot_r2_comparison_by_model(ASSET, results_df)

    """
Cross-Asset Forecasting — TCN / Conv1D-LSTM / ConvLSTM-style, corrected.

Run this once per asset by changing ASSET below. Same script, same fixes,
for Gold, Crude Oil, Forex, and NIFTY 50 — that's the point of parameterizing
it instead of hand-copying per asset like the original notebook did.

Fixes applied vs. the original notebook (cells 20/21/22), all discussed and
agreed on earlier in this project:

  1. TCN: dilation schedule trimmed from [1,2,4,8] (~61-day receptive field)
     to [1,2,4] (~29 days) to match LOOKBACK=25, and GlobalAveragePooling1D
     (which averaged away recency) replaced with taking the last timestep.
  2. Conv1D-LSTM / ConvLSTM-style: added dilation to the Conv1D layers
     (receptive field ~5 days -> ~15 days with a 3-layer dilated stack),
     and BatchNormalization replaced with LayerNormalization (robust to
     shuffle=False + heavily overlapping sliding-window batches).
  3. Data loading is now self-contained per run: DATA_PATH is actually read
     with pd.read_csv() every time (previously set but unused dead code).
  4. Missing/unusable values in the raw source are removed via dropna after
     numeric coercion — confirmed as the intended, disclosed cleaning
     approach, not an artifact of how the data was queried. Gaps in the
     resulting date index (up to several weeks in Gold/Forex/Crude Oil) are
     expected and are handled by the history-aware sequence builder below,
     not treated as an error.
  5. Chronological train/val/test split happens BEFORE scaling, and the
     MinMaxScaler is fit on the training portion of each regime only.
  6. Regime boundaries are year-based (2007-2011 / 2012-2019 / 2020-2022 /
     2023-present), which works unmodified across all four assets despite
     their slightly different start dates (Gold/Crude/Forex ~Jan 2007,
     NIFTY Sept 2007) and their common end date (~Nov 2025).
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, LSTM, Dense, Dropout, Add, Activation, LayerNormalization, Lambda
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# 1. CONFIG — pick the asset for this run
# ============================================================

ASSET_CONFIGS = {
    "Gold":      {"path": "gold_2007_to_present.csv",       "date_range": "2007-01-02 -> 2025-11-09"},
    "Crude Oil": {"path": "crude_oil_2007_to_present.csv",  "date_range": "2007-01-02 -> 2025-11-09"},
    "Forex":     {"path": "forex_usd_inr_2007_to_present.csv", "date_range": "2007-01-01 -> 2025-11-10"},
    "NIFTY 50":  {"path": "nifty50_2007_to_now.csv",        "date_range": "2007-09-17 -> 2025-11-25"},
}

ASSET = "NIFTY 50"   # <-- change this (and re-run) for each of the four assets
DATA_PATH = ASSET_CONFIGS[ASSET]["path"]

LOOKBACK = 25
FEATURES = ["Open", "High", "Low", "Close"]
EPOCHS = 100
BATCH_SIZE = 32
VAL_FRACTION = 0.15   # of each regime's pre-test data
TEST_FRACTION = 0.15  # of each regime

REGIMES = {
    "DF1": (2007, 2011),
    "DF2": (2012, 2019),
    "DF3": (2020, 2022),
    "DF4": (2023, None),   # None = open-ended, up to the max date actually in the file
}

# ============================================================
# 2. DATA LOADING — always reloaded fresh, never inherited from a stale df
# ============================================================

def load_asset_data(data_path):
    df = pd.read_csv(data_path)
    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
    df = df.dropna(subset=["Date"]).sort_values("Date").drop_duplicates(subset=["Date"], keep="first")

    for col in FEATURES:
        if col not in df.columns:
            raise ValueError(f"Expected column '{col}' not found. Columns present: {df.columns.tolist()}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    before = len(df)
    df = df.dropna(subset=FEATURES).reset_index(drop=True)   # remove rows with unusable/missing values
    removed = before - len(df)

    print(f"[{ASSET}] Loaded {data_path}: {len(df)} rows kept ({removed} rows removed for missing/invalid values)")
    print(f"[{ASSET}] Date range in file: {df['Date'].min().date()} -> {df['Date'].max().date()}")
    return df


def regime_slice(df, start_year, end_year):
    mask = df["Date"].dt.year >= start_year
    if end_year is not None:
        mask &= df["Date"].dt.year <= end_year
    return df[mask].reset_index(drop=True)


# ============================================================
# 3. TARGET CONSTRUCTION — next-observation High/Low as a ratio to Close
# ============================================================

def add_ratio_targets(regime_df):
    regime_df = regime_df.copy()
    regime_df["Target_High_Ratio"] = regime_df["High"].shift(-1) / regime_df["Close"]
    regime_df["Target_Low_Ratio"] = regime_df["Low"].shift(-1) / regime_df["Close"]
    return regime_df.dropna(subset=["Target_High_Ratio", "Target_Low_Ratio"]).reset_index(drop=True)


# ============================================================
# 4. CHRONOLOGICAL SPLIT + SCALING (fit on train only) + HISTORY-AWARE SEQUENCES
# ============================================================

def chronological_split(regime_df):
    n = len(regime_df)
    n_test = int(n * TEST_FRACTION)
    n_trainval = n - n_test
    n_val = int(n_trainval * VAL_FRACTION)
    n_train = n_trainval - n_val

    train_df = regime_df.iloc[:n_train].reset_index(drop=True)
    val_df = regime_df.iloc[n_train:n_train + n_val].reset_index(drop=True)
    test_df = regime_df.iloc[n_train + n_val:].reset_index(drop=True)
    return train_df, val_df, test_df


def create_sequences_with_history(history_X, history_close, split_X, split_y, split_close, lookback):
    """Builds sequences for a split using the tail of the PRECEDING split as
    lookback history, so the first `lookback` rows of val/test aren't wasted."""
    combined_X = np.concatenate([history_X[-lookback:], split_X], axis=0)
    X_seq, y_seq, close_seq = [], [], []
    for i in range(len(split_X)):
        X_seq.append(combined_X[i:i + lookback])
        y_seq.append(split_y[i])
        close_seq.append(split_close[i])
    return np.array(X_seq), np.array(y_seq), np.array(close_seq)


def prepare_regime(regime_df):
    train_df, val_df, test_df = chronological_split(regime_df)

    scaler = MinMaxScaler()
    scaler.fit(train_df[FEATURES])   # train-only fit — no leakage from val/test

    train_X = scaler.transform(train_df[FEATURES])
    val_X = scaler.transform(val_df[FEATURES])
    test_X = scaler.transform(test_df[FEATURES])

    train_y = train_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    val_y = val_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    test_y = test_df[["Target_High_Ratio", "Target_Low_Ratio"]].values

    # Plain (non-history) sequences for train:
    X_train, y_train = [], []
    for i in range(LOOKBACK, len(train_X)):
        X_train.append(train_X[i - LOOKBACK:i])
        y_train.append(train_y[i])
    X_train, y_train = np.array(X_train), np.array(y_train)
    close_train = train_df["Close"].values[LOOKBACK:]

    # History-aware sequences for val/test:
    X_val, y_val, close_val = create_sequences_with_history(
        train_X, train_df["Close"].values, val_X, val_y, val_df["Close"].values, LOOKBACK
    )
    X_test, y_test, close_test = create_sequences_with_history(
        np.concatenate([train_X, val_X], axis=0), np.concatenate([train_df["Close"].values, val_df["Close"].values]),
        test_X, test_y, test_df["Close"].values, LOOKBACK
    )

    return {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val, "close_val": close_val,
        "X_test": X_test, "y_test": y_test, "close_test": close_test,
        "dates_test": test_df["Date"].values,
        "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    }


# ============================================================
# 5. CORRECTED MODEL ARCHITECTURES
# ============================================================

TCN_FILTERS = 64
CONV_FILTERS = 64
KERNEL_SIZE = 3
TCN_DILATIONS = [1, 2, 4]      # trimmed from [1,2,4,8]; receptive field ~29 days, matches LOOKBACK=25
DROPOUT = 0.20
DENSE_UNITS = 16
LEARNING_RATE = 0.001


def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    residual = x
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding="causal",
               dilation_rate=dilation_rate, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding="causal",
               dilation_rate=dilation_rate, activation=None)(x)
    x = Dropout(dropout_rate)(x)
    if residual.shape[-1] != filters:
        residual = Conv1D(filters=filters, kernel_size=1, padding="same")(residual)
    x = Add()([x, residual])
    return Activation("relu")(x)


def build_tcn_model(input_shape):
    inputs = Input(shape=input_shape)
    x = inputs
    for dilation in TCN_DILATIONS:
        x = tcn_residual_block(x, TCN_FILTERS, KERNEL_SIZE, dilation, DROPOUT)
    x = Lambda(lambda t: t[:, -1, :])(x)          # last timestep, not GlobalAveragePooling1D
    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="TCN")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse", metrics=["mae"])
    return model


def build_conv1d_lstm_model(input_shape):
    inputs = Input(shape=input_shape)
    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=1)(inputs)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=2)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=4)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(DROPOUT)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="Conv1D-LSTM")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    return model


def build_convlstm_style_model(input_shape):
    # Same fixes as Conv1D-LSTM; kept as a separate configuration (ordering/
    # depth style) so the two remain a genuine ablation rather than a
    # duplicate, as agreed on earlier.
    inputs = Input(shape=input_shape)
    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=1, activation=None)(inputs)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=2, activation=None)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(DROPOUT)(x)

    x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding="causal", dilation_rate=4, activation=None)(x)
    x = LayerNormalization()(x)
    x = Activation("relu")(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(DROPOUT)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs, name="ConvLSTM-style")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    return model


MODEL_BUILDERS = {
    "TCN": build_tcn_model,
    "Conv1D-LSTM": build_conv1d_lstm_model,
    "ConvLSTM-style": build_convlstm_style_model,
}

# ============================================================
# 6. METRICS — computed on reconstructed absolute price, not the ratio scale
# ============================================================

def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}


def train_and_evaluate(model_name, regime_name, data):
    builder = MODEL_BUILDERS[model_name]
    model = builder(input_shape=(LOOKBACK, len(FEATURES)))

    early_stop = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
    model.fit(
        data["X_train"], data["y_train"],
        validation_data=(data["X_val"], data["y_val"]),
        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=False,
        callbacks=[early_stop], verbose=0,
    )

    pred_ratio = model.predict(data["X_test"], verbose=0)
    pred_high = pred_ratio[:, 0] * data["close_test"]
    pred_low = pred_ratio[:, 1] * data["close_test"]

    true_high = data["y_test"][:, 0] * data["close_test"]
    true_low = data["y_test"][:, 1] * data["close_test"]

    metrics_high = compute_metrics(true_high, pred_high)
    metrics_low = compute_metrics(true_low, pred_low)

    result = {
        "Asset": ASSET, "Model": model_name, "Regime": regime_name,
        "R2_High": metrics_high["R2"], "R2_Low": metrics_low["R2"],
        "MAE_High": metrics_high["MAE"], "MAE_Low": metrics_low["MAE"],
        "RMSE_High": metrics_high["RMSE"], "RMSE_Low": metrics_low["RMSE"],
        "MAPE_High": metrics_high["MAPE"], "MAPE_Low": metrics_low["MAPE"],
        "n_train": data["n_train"], "n_val": data["n_val"], "n_test": data["n_test"],
    }

    predictions = {
        "dates": data["dates_test"],
        "true_high": true_high, "pred_high": pred_high,
        "true_low": true_low, "pred_low": pred_low,
        "r2_high": metrics_high["R2"], "r2_low": metrics_low["R2"],
    }

    return result, predictions


REGIME_ORDER = ["DF1", "DF2", "DF3", "DF4"]


def plot_predicted_vs_actual_by_model(asset_name, all_predictions):
    """
    One figure PER MODEL (TCN, Conv1D-LSTM, ConvLSTM-style), with all
    available regimes (DF1..DF4) as subplots inside it — clubbed by model,
    not one figure per regime with all models crammed inside it.

    Each subplot shows actual vs. predicted High and Low over that regime's
    test period, with R² in the subplot title. Saves one PNG per model.
    """
    for model_name, regime_preds in all_predictions.items():
        regimes_present = [r for r in REGIME_ORDER if r in regime_preds]
        if not regimes_present:
            continue

        n = len(regimes_present)
        n_cols = 2 if n > 1 else 1
        n_rows = int(np.ceil(n / n_cols))

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 4.5 * n_rows), squeeze=False)
        fig.suptitle(f"{asset_name} — {model_name}: Predicted vs. Actual (all regimes)", fontsize=14, y=1.02)

        for idx, regime_name in enumerate(regimes_present):
            ax = axes[idx // n_cols][idx % n_cols]
            p = regime_preds[regime_name]

            ax.plot(p["dates"], p["true_high"], label="Actual High", color="tab:blue", linewidth=1.2)
            ax.plot(p["dates"], p["pred_high"], label="Predicted High", color="tab:blue", linewidth=1.0, linestyle="--")
            ax.plot(p["dates"], p["true_low"], label="Actual Low", color="tab:orange", linewidth=1.2)
            ax.plot(p["dates"], p["pred_low"], label="Predicted Low", color="tab:orange", linewidth=1.0, linestyle="--")

            ax.set_title(f"{regime_name}  (R\u00b2 High={p['r2_high']:.3f}, Low={p['r2_low']:.3f})", fontsize=11)
            ax.set_xlabel("Date")
            ax.set_ylabel("Price")
            ax.tick_params(axis="x", rotation=30)
            if idx == 0:
                ax.legend(fontsize=8, loc="best")

        # hide any unused subplot slots
        for idx in range(n, n_rows * n_cols):
            axes[idx // n_cols][idx % n_cols].axis("off")

        plt.tight_layout()
        filename = f"{asset_name.replace(' ', '_').lower()}_{model_name.replace(' ', '_').lower()}_predicted_vs_actual.png"
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {filename}")


def plot_r2_comparison_by_model(asset_name, results_df):
    """
    Other relevant plot: a single bar chart per model showing R² (High and
    Low) across DF1..DF4 side by side — a compact companion to the line
    plots above for quickly comparing regime difficulty within one model.
    """
    for model_name in results_df["Model"].unique():
        sub = results_df[results_df["Model"] == model_name].set_index("Regime").reindex(REGIME_ORDER).dropna(how="all")
        if sub.empty:
            continue

        fig, ax = plt.subplots(figsize=(7, 4))
        x = np.arange(len(sub))
        width = 0.35
        ax.bar(x - width / 2, sub["R2_High"], width, label="R² High", color="tab:blue")
        ax.bar(x + width / 2, sub["R2_Low"], width, label="R² Low", color="tab:orange")
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(sub.index)
        ax.set_ylabel("R²")
        ax.set_title(f"{asset_name} — {model_name}: R² by Regime")
        ax.legend()
        plt.tight_layout()
        filename = f"{asset_name.replace(' ', '_').lower()}_{model_name.replace(' ', '_').lower()}_r2_by_regime.png"
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {filename}")


# ============================================================
# 8. MAIN LOOP — all regimes x all three corrected architectures
# ============================================================

if __name__ == "__main__":
    df = load_asset_data(DATA_PATH)
    results = []
    # all_predictions[model_name][regime_name] = predictions dict — grouped by
    # MODEL first, so plotting can club all four regimes under one model,
    # not one figure per regime with models inside it.
    all_predictions = {model_name: {} for model_name in MODEL_BUILDERS}

    for regime_name, (start_year, end_year) in REGIMES.items():
        regime_df = regime_slice(df, start_year, end_year)
        regime_df = add_ratio_targets(regime_df)

        min_required = LOOKBACK + 20
        if len(regime_df) <= min_required:
            print(f"[{ASSET}] Skipping {regime_name}: only {len(regime_df)} usable rows (< {min_required}).")
            continue

        data = prepare_regime(regime_df)
        print(f"\n[{ASSET}] {regime_name}: train={data['n_train']}, val={data['n_val']}, test={data['n_test']}")

        for model_name in MODEL_BUILDERS:
            print(f"  Training {model_name}...")
            result, predictions = train_and_evaluate(model_name, regime_name, data)
            results.append(result)
            all_predictions[model_name][regime_name] = predictions
            print(f"    R2 High/Low: {result['R2_High']:.3f} / {result['R2_Low']:.3f}")

    results_df = pd.DataFrame(results)
    print(f"\n=== {ASSET} — Full Results ===")
    print(results_df.to_string(index=False))
    results_df.to_csv(f"{ASSET.replace(' ', '_').lower()}_tcn_convlstm_results.csv", index=False)

    plot_predicted_vs_actual_by_model(ASSET, all_predictions)
    plot_r2_comparison_by_model(ASSET, results_df)

[Forex] Loaded forex_usd_inr_2007_to_present.csv: 4889 rows kept (0 rows removed for missing/invalid values)
[Forex] Date range in file: 2007-01-01 -> 2025-11-10

[Forex] DF1: train=925, val=163, test=191
  Training TCN...
    R2 High/Low: 0.939 / 0.937
  Training Conv1D-LSTM...
    R2 High/Low: 0.986 / 0.972
  Training ConvLSTM-style...
    R2 High/Low: 0.971 / 0.974

[Forex] DF2: train=1505, val=265, test=312
  Training TCN...
    R2 High/Low: -11.248 / -9.516
  Training Conv1D-LSTM...
    R2 High/Low: 0.807 / 0.875
  Training ConvLSTM-style...
    R2 High/Low: 0.418 / -2.203

[Forex] DF3: train=566, val=99, test=117
  Training TCN...
    R2 High/Low: -57.446 / -39.576
  Training Conv1D-LSTM...
    R2 High/Low: 0.918 / -0.062
  Training ConvLSTM-style...
    R2 High/Low: 0.909 / 0.935

[Forex] DF4: train=537, val=94, test=111
  Training TCN...
    R2 High/Low: -43.826 / -36.290
  Training Conv1D-LSTM...
    R2 High/Low: 0.648 / 0.745
  Training ConvLSTM-style...
    R2 High/Low: 0.86

In [5]:
# Research review: This saved ablation changes training protocol and architecture. Treat it as one historical experiment, with its exact test dates, seeds and sampling, rather than proof of a universal architectural improvement.
"""
Ablation harness — isolates WHY ConvLSTM-style's Gold/DF1 R^2 dropped from
~0.95 (original notebook) to ~0.62 (corrected pipeline).

Two independent things changed between those two versions:
  A) The TRAINING PROTOCOL: original had a plain train/test split and a
     fixed epoch count; the corrected version added a train/val/test split
     plus EarlyStopping(patience=15) monitoring val_loss.
  B) The ARCHITECTURE: original used BatchNormalization + non-dilated
     Conv1D (kernel=3, no dilation); corrected uses LayerNormalization +
     dilated Conv1D (dilation_rate 1,2,4).

This script runs all four combinations of {A on/off} x {B on/off} on the
SAME data (Gold, DF1) so you can see, numerically, which one actually moved
the R^2 — instead of attributing the whole drop to "the fix."

Run this as-is (after setting GOLD_CSV_PATH below). It trains 4 models
(each ~1-2 min on CPU for a regime this size) and prints a comparison table
at the end.
"""

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, LSTM, Dense, Dropout, BatchNormalization, LayerNormalization, Activation
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

SEED = 42
GOLD_CSV_PATH = "gold_2007_to_present.csv"   # <-- set this to your actual file
LOOKBACK = 25
FEATURES = ["Open", "High", "Low", "Close"]
EPOCHS = 100
BATCH_SIZE = 32
CONV_FILTERS = 64
KERNEL_SIZE = 3
DROPOUT = 0.20
DENSE_UNITS = 16
LEARNING_RATE = 0.001


# ============================================================
# Data loading (same as the corrected pipeline)
# ============================================================

def load_asset_data(data_path):
    df = pd.read_csv(data_path)
    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
    df = df.dropna(subset=["Date"]).sort_values("Date").drop_duplicates(subset=["Date"], keep="first")
    for col in FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=FEATURES).reset_index(drop=True)
    return df


def regime_slice(df, start_year, end_year):
    mask = df["Date"].dt.year >= start_year
    if end_year is not None:
        mask &= df["Date"].dt.year <= end_year
    return df[mask].reset_index(drop=True)


def add_ratio_targets(regime_df):
    regime_df = regime_df.copy()
    regime_df["Target_High_Ratio"] = regime_df["High"].shift(-1) / regime_df["Close"]
    regime_df["Target_Low_Ratio"] = regime_df["Low"].shift(-1) / regime_df["Close"]
    return regime_df.dropna(subset=["Target_High_Ratio", "Target_Low_Ratio"]).reset_index(drop=True)


def create_sequences_with_history(history_X, split_X, split_y, split_close, lookback):
    combined_X = np.concatenate([history_X[-lookback:], split_X], axis=0)
    X_seq, y_seq, close_seq = [], [], []
    for i in range(len(split_X)):
        X_seq.append(combined_X[i:i + lookback])
        y_seq.append(split_y[i])
        close_seq.append(split_close[i])
    return np.array(X_seq), np.array(y_seq), np.array(close_seq)


# ============================================================
# Two data-prep modes: OLD-STYLE (train/test only) vs NEW-STYLE (train/val/test)
# ============================================================

def prepare_old_style(regime_df, test_fraction=0.15):
    """Matches the original notebook: one chronological split, no validation
    slice carved out, scaler fit on the train portion."""
    n = len(regime_df)
    n_test = int(n * test_fraction)
    n_train = n - n_test
    train_df = regime_df.iloc[:n_train].reset_index(drop=True)
    test_df = regime_df.iloc[n_train:].reset_index(drop=True)

    scaler = MinMaxScaler()
    scaler.fit(train_df[FEATURES])
    train_X = scaler.transform(train_df[FEATURES])
    test_X = scaler.transform(test_df[FEATURES])
    train_y = train_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    test_y = test_df[["Target_High_Ratio", "Target_Low_Ratio"]].values

    X_train, y_train = [], []
    for i in range(LOOKBACK, len(train_X)):
        X_train.append(train_X[i - LOOKBACK:i])
        y_train.append(train_y[i])
    X_train, y_train = np.array(X_train), np.array(y_train)

    X_test, y_test, close_test = create_sequences_with_history(
        train_X, test_X, test_y, test_df["Close"].values, LOOKBACK
    )

    return {
        "X_train": X_train, "y_train": y_train,
        "X_val": None, "y_val": None,
        "X_test": X_test, "y_test": y_test, "close_test": close_test,
        "n_train": n_train, "n_val": 0, "n_test": len(test_df),
    }


def prepare_new_style(regime_df, val_fraction=0.15, test_fraction=0.15):
    """Matches the corrected pipeline: train/val/test, scaler fit on train only."""
    n = len(regime_df)
    n_test = int(n * test_fraction)
    n_trainval = n - n_test
    n_val = int(n_trainval * val_fraction)
    n_train = n_trainval - n_val

    train_df = regime_df.iloc[:n_train].reset_index(drop=True)
    val_df = regime_df.iloc[n_train:n_train + n_val].reset_index(drop=True)
    test_df = regime_df.iloc[n_train + n_val:].reset_index(drop=True)

    scaler = MinMaxScaler()
    scaler.fit(train_df[FEATURES])
    train_X = scaler.transform(train_df[FEATURES])
    val_X = scaler.transform(val_df[FEATURES])
    test_X = scaler.transform(test_df[FEATURES])
    train_y = train_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    val_y = val_df[["Target_High_Ratio", "Target_Low_Ratio"]].values
    test_y = test_df[["Target_High_Ratio", "Target_Low_Ratio"]].values

    X_train, y_train = [], []
    for i in range(LOOKBACK, len(train_X)):
        X_train.append(train_X[i - LOOKBACK:i])
        y_train.append(train_y[i])
    X_train, y_train = np.array(X_train), np.array(y_train)

    X_val, y_val, _ = create_sequences_with_history(train_X, val_X, val_y, val_df["Close"].values, LOOKBACK)
    X_test, y_test, close_test = create_sequences_with_history(
        np.concatenate([train_X, val_X], axis=0), test_X, test_y, test_df["Close"].values, LOOKBACK
    )

    return {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "X_test": X_test, "y_test": y_test, "close_test": close_test,
        "n_train": n_train, "n_val": n_val, "n_test": len(test_df),
    }


# ============================================================
# ConvLSTM-style, with normalization and dilation as toggles
# ============================================================

def build_convlstm_style_model(input_shape, use_layernorm, use_dilation):
    norm_layer = LayerNormalization if use_layernorm else BatchNormalization
    dilations = [1, 2, 4] if use_dilation else [1, 1, 1]
    padding = "causal" if use_dilation else "same"   # original used 'same'; causal only matters once dilation > 1

    inputs = Input(shape=input_shape)
    x = inputs
    for d in dilations:
        x = Conv1D(CONV_FILTERS, KERNEL_SIZE, padding=padding, dilation_rate=d, activation=None)(x)
        x = norm_layer()(x)
        x = Activation("relu")(x)
        x = Dropout(DROPOUT)(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(DROPOUT)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(DENSE_UNITS, activation="relu")(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    return model


# ============================================================
# Metrics
# ============================================================

def compute_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
    }


def run_config(regime_df, config_name, use_val_split, use_early_stopping, use_layernorm, use_dilation):
    tf.random.set_seed(SEED)
    np.random.seed(SEED)

    data = prepare_new_style(regime_df) if use_val_split else prepare_old_style(regime_df)

    model = build_convlstm_style_model(
        input_shape=(LOOKBACK, len(FEATURES)),
        use_layernorm=use_layernorm,
        use_dilation=use_dilation,
    )

    fit_kwargs = dict(epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=False, verbose=0)
    if use_val_split and use_early_stopping:
        fit_kwargs["validation_data"] = (data["X_val"], data["y_val"])
        fit_kwargs["callbacks"] = [EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)]
    elif use_val_split:
        # validation data present but NOT used for early stopping — trains the full fixed epoch count
        fit_kwargs["validation_data"] = (data["X_val"], data["y_val"])

    model.fit(data["X_train"], data["y_train"], **fit_kwargs)

    pred_ratio = model.predict(data["X_test"], verbose=0)
    pred_high = pred_ratio[:, 0] * data["close_test"]
    true_high = data["y_test"][:, 0] * data["close_test"]
    pred_low = pred_ratio[:, 1] * data["close_test"]
    true_low = data["y_test"][:, 1] * data["close_test"]

    m_high = compute_metrics(true_high, pred_high)
    m_low = compute_metrics(true_low, pred_low)

    return {
        "Config": config_name,
        "Val split used": use_val_split,
        "Early stopping used": use_early_stopping,
        "Normalization": "LayerNorm" if use_layernorm else "BatchNorm",
        "Dilation": "1,2,4" if use_dilation else "none (kernel=3 only)",
        "n_train": data["n_train"], "n_val": data["n_val"], "n_test": data["n_test"],
        "R2_High": m_high["R2"], "R2_Low": m_low["R2"],
        "MAE_High": m_high["MAE"], "MAE_Low": m_low["MAE"],
    }


if __name__ == "__main__":
    df = load_asset_data(GOLD_CSV_PATH)
    df1 = regime_slice(df, 2007, 2011)
    df1 = add_ratio_targets(df1)
    print(f"Gold DF1: {len(df1)} usable rows")

    configs = [
        # (name, use_val_split, use_early_stopping, use_layernorm, use_dilation)
        ("1. Original-style (train/test, fixed epochs, BatchNorm, no dilation)", False, False, False, False),
        ("2. Architecture fix only (train/test, fixed epochs, LayerNorm, dilation)", False, False, True, True),
        ("3. Training-protocol change only (val split + early stop, BatchNorm, no dilation)", True, True, False, False),
        ("4. Full corrected pipeline (val split + early stop, LayerNorm, dilation)", True, True, True, True),
    ]

    results = []
    for name, use_val, use_es, use_ln, use_dil in configs:
        print(f"\nRunning: {name}")
        result = run_config(df1, name, use_val, use_es, use_ln, use_dil)
        results.append(result)
        print(f"  R2 High/Low: {result['R2_High']:.3f} / {result['R2_Low']:.3f}")

    results_df = pd.DataFrame(results)
    print("\n=== Ablation Results: Gold / DF1 / ConvLSTM-style ===")
    print(results_df.to_string(index=False))
    results_df.to_csv("ablation_gold_df1_convlstm_style.csv", index=False)

    print("""
How to read this:
  Compare row 1 (closest to your original notebook) to rows 2 and 3 to see
  whether the architecture fix (row 2) or the training-protocol change
  (row 3) accounts for more of the gap to row 1. Row 4 is the corrected
  pipeline you already ran. If row 3's R2 is much closer to row 1 than
  row 2's is, the validation-split/early-stopping addition is the main
  cause of the drop, not the LayerNorm/dilation fix -- in which case the
  fix itself is fine, and the training protocol (e.g. patience, or just
  running the full fixed epoch count) is what needs adjusting.
""")

Gold DF1: 1260 usable rows

Running: 1. Original-style (train/test, fixed epochs, BatchNorm, no dilation)
  R2 High/Low: 0.921 / 0.885

Running: 2. Architecture fix only (train/test, fixed epochs, LayerNorm, dilation)
  R2 High/Low: 0.821 / 0.936

Running: 3. Training-protocol change only (val split + early stop, BatchNorm, no dilation)
  R2 High/Low: 0.763 / 0.933

Running: 4. Full corrected pipeline (val split + early stop, LayerNorm, dilation)
  R2 High/Low: 0.936 / 0.958

=== Ablation Results: Gold / DF1 / ConvLSTM-style ===
                                                                           Config  Val split used  Early stopping used Normalization             Dilation  n_train  n_val  n_test  R2_High   R2_Low  MAE_High   MAE_Low
             1. Original-style (train/test, fixed epochs, BatchNorm, no dilation)           False                False     BatchNorm none (kernel=3 only)     1071      0     189 0.921297 0.885160 30.711459 31.217106
         2. Architecture fix only